# Can Review Text Predict a Restaurant's Future Quality Decline?

**Yelp-Native Future Restaurant Risk Prediction: Supervised Learning with Imbalanced Labels**

### The Core Objective
Task 6 is the capstone prediction task. Using everything accumulated across Tasks 1–5 — Yelp reviews, business metadata, dish vocabulary, sentiment signals, temporal features, and text representations — you will build and evaluate a supervised classifier that predicts whether a restaurant is at risk of a measurable quality decline in a future time window. Unlike Tasks 3–5, which focus on Chinese cuisine, Task 6 draws from **all Yelp restaurant categories** — subsetting to a single cuisine would leave too few labeled examples for reliable cross-validation, so broad cuisine vocabulary (sushi, steak, tacos, beer) appearing in top features is expected and reflects that design choice, not a mistake.

This is a *genuinely hard* prediction problem. The labels are noisy, the positive class is rare, the feature set is high-dimensional, and the right evaluation metric (macro-F1) penalizes you equally for missing positive examples and over-predicting them. That difficulty is the point.

### Our Analytical Approach

1. **The Baseline:** Dummy classifier and metadata-only model — how well can you do without any text?
2. **Text-Only Classifier:** TF-IDF + Logistic Regression or SVM — what signals does the review vocabulary carry?
3. **Text + Metadata Model:** Combine TF-IDF features with structured fields (category, rating history, temporal features).
4. **Advanced Representation:** Sentence embeddings, pooled review embeddings, class-weighted models, or ensemble methods.
5. **Evaluation Framework:** Stratified K-fold CV, macro-F1, per-class precision/recall, and failure analysis.
6. **Leakage Audit:** Verify that no future-derived feature contaminates the prediction frame.

### Why This Matters
This task teaches you a lesson that every applied ML practitioner eventually learns: *building the model is the easy part; building the right labels, auditing for leakage, and choosing the right evaluation metric are what determine whether a model is actually useful*. The main metric here is **macro-F1** — it gives equal weight to both classes, so a model that only predicts "no risk" will score 0.50 or below. You have to actually find the at-risk restaurants.


## Table of Contents

1. [Learning Objectives](#Learning-objectives)
2. [Required Artifact Contract](#Required-artifact-contract-for-Task-6)
3. [Configuration](#1.-Configuration)
4. [Load Yelp-Shaped Data](#2.-Load-Yelp-Shaped-Data)
5. [Build the Prediction Frame](#3.-Build-the-Future-Quality-Decline-Prediction-Frame)
6. [Feature Groups and Model Pipelines](#5.-Feature-Groups-and-Model-Pipelines)
7. [Cross-Validated Model Comparison](#6.-Cross-Validated-Model-Comparison)
8. [Selected Model Diagnostics](#7.-Selected-Model-Diagnostics-and-Error-Analysis)
9. [Feature Importance](#8.-Feature-Importance-or-Coefficients)
10. [Leakage Audit](#9.-Leakage-Audit)
11. [Visual Evidence and Interpretation](#Visual-Evidence-and-Interpretation)
12. [Required Report Deliverables](#Required-Report-Deliverables)


# Task 6 — Yelp-Native Future Restaurant Risk Prediction

The main metric is **macro-F1**, supported by precision, recall, per-class F1, and confusion-matrix diagnostics.

## Learning objectives

By completing this task, you should be able to:

1. Frame a leakage-safe future prediction problem with historical features and future labels.
2. Build and inspect structured numeric features from historical review behavior.
3. Compare metadata-only, text-only, and text+metadata classifiers.
4. Evaluate imbalanced classification using macro-F1 and confusion matrices.
5. Audit temporal leakage and interpret failure cases.


## Required artifact contract for Task 6

Your completed notebook should produce these core artifacts under `artifacts/task6/`:

- `business_prediction_frame.csv` — one row per eligible business with historical features
  and future risk label.
- `class_balance.csv` — class counts and risk-rate diagnostics.
- `model_comparison_results.csv` — cross-validated metrics including macro-F1.
- `selected_model_classification_report.csv` — selected model diagnostics.
- `confusion_matrix_selected_model.csv` — confusion matrix for error analysis.
- `error_analysis_cases.csv` — error analysis outputs.
- `leakage_audit.json` — explicit temporal leakage audit.

In [ ]:
import os
import re
import json
import math
import random
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
print("Standard library and data analysis imports complete.")

### Why These Imports Matter

**scikit-learn components**: We use `ColumnTransformer` to apply different preprocessing to text, numeric, and categorical features; `Pipeline` to chain preprocessing and model training; `TfidfVectorizer` for text representation; `TruncatedSVD` for dimensionality reduction; and `StratifiedKFold` for balanced cross-validation.

**Key insight**: Task 6 is a **supervised learning** endpoint. Unlike Tasks 1-5 (exploratory data mining), we now have labeled examples (risk vs non-risk businesses) and must prevent leakage between historical features and future labels.

## About Macro-F1 for Imbalanced Evaluation

Macro-F1 averages F1 scores across classes, giving equal weight to the majority and minority class. In Task 6, it is the primary metric because accuracy can look high even when a model ignores the rare risk/fail class. Macro-F1 forces students to care about both pass and risk prediction quality.

This metric should be read alongside per-class precision/recall and confusion matrices. A model with slightly lower accuracy but better minority-class recall may be more informative for the supervised-learning objective. You should explain metric tradeoffs rather than only choosing the largest number in the table.

**Key Resources:**
- [scikit-learn f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) - Official API.
- [scikit-learn classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics) - Metric guidance.
- [Wikipedia: F-score](https://en.wikipedia.org/wiki/F-score) - Conceptual overview.

In [ ]:
from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("scikit-learn model and evaluation imports complete.")

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
print("Random seeds set for reproducibility.")

In [ ]:
# Xiaofei's code — Task 6 local path bootstrap / 本地真实数据与输出路径配置
# 中文注释：这个 cell 要在 Section 1 Configuration 之前运行，用来避免最终预测任务误用 smoke_test。

# 中文注释：重新导入 os，确保单独运行本 cell 时也能设置环境变量。
import os
# 中文注释：重新导入 Path，确保单独运行本 cell 时也能处理路径。
from pathlib import Path

# 中文注释：定义项目根目录查找函数，避免从 notebooks 子目录运行时读写到错误位置。
def find_project_root(start: Path) -> Path:
    # 中文注释：从当前目录开始逐级向上检查候选目录。
    for candidate in [start, *start.parents]:
        # 中文注释：项目根目录应包含 notebooks 目录。
        has_notebooks = (candidate / "notebooks").is_dir()
        # 中文注释：最终分析必须使用真实 Yelp 数据目录。
        has_raw_data = (candidate / "data" / "raw").is_dir()
        # 中文注释：本项目通常包含本地虚拟环境或项目包目录。
        has_project_package = (candidate / "cs598capstone").is_dir()
        # 中文注释：同时具备 notebooks 且具备真实数据或项目包时，认为是项目根目录。
        if has_notebooks and (has_raw_data or has_project_package):
            # 中文注释：返回识别出的项目根目录。
            return candidate
    # 中文注释：如果没有找到严格匹配的根目录，则退回到启动目录。
    return start

# 中文注释：记录前面初始化 cell 可能已经创建的旧输出目录。
bootstrap_artifact_dir = globals().get("ARTIFACT_DIR")
# 中文注释：优先读取显式配置的项目根目录。
env_root = os.environ.get("DMC_PROJECT_ROOT", "").strip()
# 中文注释：如果用户配置了项目根目录，则使用该路径。
if env_root:
    # 中文注释：展开并规范化用户提供的项目根目录。
    PROJECT_ROOT = Path(env_root).expanduser().resolve()
# 中文注释：否则从当前工作目录自动向上查找项目根目录。
else:
    # 中文注释：根据当前目录自动识别项目根目录。
    PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# 中文注释：切换到项目根目录，保证相对路径都指向项目内的数据和 artifacts。
os.chdir(PROJECT_ROOT)

# 中文注释：强制使用真实本地 Yelp 数据，不使用 smoke_test。
os.environ["DMC_DATA_SOURCE"] = "manual"
# 中文注释：同步设置 Task 6 专用数据来源变量，避免旧环境变量回落到 smoke_test。
os.environ["TASK6_DATA_SOURCE"] = "manual"
# 中文注释：指定真实 Yelp JSON 数据目录。
os.environ["DMC_DATA_DIR"] = str(PROJECT_ROOT / "data" / "raw")
# 中文注释：同步设置 Task 6 专用数据目录变量。
os.environ["TASK6_DATA_DIR"] = str(PROJECT_ROOT / "data" / "raw")
# 中文注释：本地 round-4 Yelp 数据的评论日期截至 2014-07-16，因此使用 2013-07-01 作为历史/未来切分点。
os.environ["TASK6_CUTOFF_DATE"] = "2013-07-01"
# 中文注释：为真实数据设置可复现且运行时间可控的餐厅数量上限。
os.environ["TASK6_MAX_BUSINESSES"] = "2000"
# 中文注释：为真实数据设置评论总量上限，当前业务样本通常会读取约六万条匹配评论。
os.environ["TASK6_MAX_REVIEWS_TOTAL"] = "120000"
# 中文注释：限制单个商家的最大评论数，避免热门商家支配模型训练。
os.environ["TASK6_MAX_REVIEWS_PER_BUSINESS"] = "180"
# 中文注释：保持三折分层交叉验证，兼顾稳定性和运行时间。
os.environ["TASK6_CV_SPLITS"] = "3"
# 中文注释：保留真实数据模式下每类至少 25 个样本的建模要求。
os.environ["TASK6_MIN_POSITIVE_EXAMPLES"] = "25"
# 中文注释：要求少数类比例至少达到 10%，避免过度不平衡的标签定义。
os.environ["TASK6_MIN_CLASS_RATE"] = "0.10"
# 中文注释：指定 Task 6 的所有输出 artifact 目录。
TASK6_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "task6"
# 中文注释：把输出目录写入 Task 6 专用环境变量。
os.environ["TASK6_ARTIFACT_DIR"] = str(TASK6_OUTPUT_DIR)
# 中文注释：同步写入通用 artifact 环境变量，保证 helper 使用同一输出位置。
os.environ["DMC_ARTIFACT_DIR"] = str(TASK6_OUTPUT_DIR)
# 中文注释：重设 notebook 全局输出目录，覆盖可能来自旧相对路径的输出目录。
ARTIFACT_DIR = TASK6_OUTPUT_DIR
# 中文注释：确保 Task 6 输出目录存在。
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 中文注释：定义本地句向量模型目录，供 embedding 分类优先离线使用。
LOCAL_EMBEDDING_MODEL_DIR = PROJECT_ROOT / "models" / "all-MiniLM-L6-v2"
# 中文注释：如果本地句向量模型目录存在，则使用本地模型避免运行时下载。
if LOCAL_EMBEDDING_MODEL_DIR.exists():
    # 中文注释：把本地 sentence-transformers 模型路径写入 Task 6 配置。
    os.environ["TASK6_EMBEDDING_MODEL_PATH"] = str(LOCAL_EMBEDDING_MODEL_DIR)
    # 中文注释：本地模型存在时关闭下载，保证结果可复现。
    os.environ["TASK6_ALLOW_EMBEDDING_DOWNLOAD"] = "0"
# 中文注释：如果本地模型不存在，则后续代码会自动使用 TF-IDF+SVD fallback。
else:
    # 中文注释：清空本地模型路径，避免误用不存在的目录。
    os.environ["TASK6_EMBEDDING_MODEL_PATH"] = ""
    # 中文注释：关闭下载并让 notebook 使用可复现的本地 fallback。
    os.environ["TASK6_ALLOW_EMBEDDING_DOWNLOAD"] = "0"

# 中文注释：如果前面 cell 创建了错误位置的空输出目录，则安全清理空目录。
if bootstrap_artifact_dir is not None:
    # 中文注释：把旧输出目录转换为 Path 对象。
    bootstrap_artifact_dir = Path(bootstrap_artifact_dir)
    # 中文注释：只有旧目录和真实输出目录不一致时才尝试清理。
    if bootstrap_artifact_dir.resolve() != ARTIFACT_DIR.resolve():
        # 中文注释：尝试删除错误位置下的空 task6 目录。
        try:
            # 中文注释：只删除空目录，目录内有文件时会自动保留。
            bootstrap_artifact_dir.rmdir()
        # 中文注释：目录不存在或非空时不影响 notebook 继续运行。
        except OSError:
            # 中文注释：跳过无法安全删除的旧目录。
            pass
        # 中文注释：尝试删除错误位置下的空 artifacts 父目录。
        try:
            # 中文注释：只删除空父目录，避免误删已有输出。
            bootstrap_artifact_dir.parent.rmdir()
        # 中文注释：父目录不存在或非空时不影响 notebook 继续运行。
        except OSError:
            # 中文注释：跳过无法安全删除的旧父目录。
            pass

# 中文注释：列出最终分析必须存在的真实 Yelp 数据文件。
required_yelp_files = [PROJECT_ROOT / "data" / "raw" / "yelp_academic_dataset_business.json", PROJECT_ROOT / "data" / "raw" / "yelp_academic_dataset_review.json"]
# 中文注释：检查真实 Yelp 数据文件是否缺失。
missing_yelp_files = [path for path in required_yelp_files if not path.exists()]
# 中文注释：如果缺少真实数据文件，则立即报错，避免回退到 smoke_test。
if missing_yelp_files:
    # 中文注释：抛出包含缺失文件路径的错误。
    raise FileNotFoundError(f"Task 6 requires real Yelp data files, missing: {missing_yelp_files}")
# 中文注释：如果数据来源仍然是 smoke_test，则立即报错。
if os.environ.get("DMC_DATA_SOURCE") == "smoke_test" or os.environ.get("TASK6_DATA_SOURCE") == "smoke_test":
    # 中文注释：禁止最终分析使用 smoke_test。
    raise ValueError("Task 6 final analysis must use manual data, not smoke_test.")

# 中文注释：打印项目根目录，便于检查路径是否正确。
print("PROJECT_ROOT:", PROJECT_ROOT)
# 中文注释：打印当前工作目录，确认已经切换到项目根目录。
print("cwd:", os.getcwd())
# 中文注释：打印真实数据来源配置。
print("DATA_SOURCE will be:", os.environ.get("DMC_DATA_SOURCE"))
# 中文注释：打印真实数据目录配置。
print("DMC_DATA_DIR will be:", os.environ.get("DMC_DATA_DIR"))
# 中文注释：打印 Task 6 cutoff 配置。
print("TASK6_CUTOFF_DATE:", os.environ.get("TASK6_CUTOFF_DATE"))
# 中文注释：打印本地 embedding 模型路径配置。
print("TASK6_EMBEDDING_MODEL_PATH:", os.environ.get("TASK6_EMBEDDING_MODEL_PATH") or "tfidf_svd_fallback")
# 中文注释：打印 Task 6 输出目录配置。
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())


## 1. Configuration

`smoke_test` creates a small Yelp-shaped temporal dataset. `manual` and `kaggle_latest` use local or Kaggle Yelp JSON files with bounded streaming. The cutoff and label thresholds are configurable so students can inspect class balance and justify target choices.

Real-data defaults intentionally use a larger bounded sample than smoke mode. Future quality decline is a relatively rare target after historical/future window filtering, so a tiny real subset can produce an unlearnable class balance even when the code runs.

In [ ]:
# Dataset mode: controls whether we use smoke/synthetic data or real Yelp data
DATASET_MODE = os.environ.get("TASK6_DATA_SOURCE", os.environ.get("DMC_DATA_SOURCE", "smoke_test"))  
DATA_ROOT = Path(os.environ.get("TASK6_DATA_DIR", os.environ.get("DMC_DATA_DIR", "data/yelp")))
KAGGLE_DATASET = os.environ.get("TASK6_KAGGLE_DATASET", "yelp-dataset/yelp-dataset")

# Restaurant filtering
REQUIRE_RESTAURANTS = os.environ.get("TASK6_REQUIRE_RESTAURANTS", "1").lower() in {"1", "true", "yes", "y"}
TARGET_CATEGORY = os.environ.get("TASK6_TARGET_CATEGORY", "")  # optional: focus on a specific cuisine

### The Critical Cutoff Date

The **cutoff date** is the dividing line between historical features (before cutoff) and future labels (after cutoff). This is the single most important leakage-control mechanism in Task 6.

**Why January 1, 2021?** This gives us historical data from 2019-2020 and future data from 2021-2023 in the Kaggle Yelp dataset. You should experiment with different cutoffs to see how it affects class balance and model viability.

In [ ]:
CUTOFF_DATE = pd.Timestamp(os.environ.get("TASK6_CUTOFF_DATE", "2021-01-01"))
REAL_DATA_MODE = DATASET_MODE != "smoke_test"

print(f"Cutoff date: {CUTOFF_DATE.date()}")
print(f"Historical window: reviews BEFORE {CUTOFF_DATE.date()}")
print(f"Future window: reviews ON OR AFTER {CUTOFF_DATE.date()}")

### Sample Size Configuration: Balancing Runtime and Class Viability

**The tension**: Predicting future quality decline is a rare-event problem. If we sample too few businesses, we might get 2 positive examples and 98 negative examples — unlearnable. If we sample too many, the notebook becomes slow.

**Smoke mode** uses 120 businesses and 12,000 reviews for quick validation. **Real mode** defaults to 2,000 businesses and 200,000 reviews to ensure we have at least 25+ positive risk examples for meaningful cross-validation.

You should understand: *configuration is not arbitrary* — it's driven by statistical requirements for the minority class.

In [ ]:
DEFAULT_MAX_BUSINESSES = 120 if not REAL_DATA_MODE else 2000
DEFAULT_MAX_REVIEWS_TOTAL = 12000 if not REAL_DATA_MODE else 200000
MAX_BUSINESSES = int(os.environ.get("TASK6_MAX_BUSINESSES", str(DEFAULT_MAX_BUSINESSES)))
MAX_REVIEWS_TOTAL = int(os.environ.get("TASK6_MAX_REVIEWS_TOTAL", str(DEFAULT_MAX_REVIEWS_TOTAL)))
MAX_REVIEWS_PER_BUSINESS = int(os.environ.get("TASK6_MAX_REVIEWS_PER_BUSINESS", "180"))

print(f"Sample limits: {MAX_BUSINESSES:,} businesses, {MAX_REVIEWS_TOTAL:,} reviews total")
print(f"Per-business cap: {MAX_REVIEWS_PER_BUSINESS} reviews")

### Window Support Requirements

Each business must have:
- **MIN_HISTORICAL_REVIEWS = 6**: Enough pre-cutoff reviews to compute reliable historical features (text, stars, sentiment)
- **MIN_FUTURE_REVIEWS = 5**: Enough post-cutoff reviews to make the future label reasonably stable

**Why 6 and 5?** These are the minimum thresholds that balance label noise and sample size. With only 3-4 future reviews, a single bad day can flip the label. With 10+ reviews, the future window becomes more reliable but we lose businesses.

In [ ]:
MIN_HISTORICAL_REVIEWS = int(os.environ.get("TASK6_MIN_HISTORICAL_REVIEWS", "6"))
MIN_FUTURE_REVIEWS = int(os.environ.get("TASK6_MIN_FUTURE_REVIEWS", "5"))

print(f"Window requirements: {MIN_HISTORICAL_REVIEWS}+ historical reviews, {MIN_FUTURE_REVIEWS}+ future reviews")

### Label Definition: What is "Risk"?

**Risk = 1** when BOTH conditions hold:
1. **Quality decline ≥ 0.75 stars**: Historical average minus future average ≥ threshold
2. **Future average ≤ 3.5 stars**: The future quality is objectively low

**Why two conditions?** A restaurant going from 5.0 to 4.3 stars (decline = 0.7) is not "at risk" — it's still excellent. But a restaurant going from 4.0 to 3.0 stars (decline = 1.0, future = 3.0) is genuinely declining into poor quality.

The notebook records how the chosen threshold affects class balance, but the core task is the leakage-safe prediction workflow.

In [ ]:
DECLINE_THRESHOLD = float(os.environ.get("TASK6_DECLINE_THRESHOLD", "0.75"))
FUTURE_MAX_AVG_STARS = float(os.environ.get("TASK6_FUTURE_MAX_AVG_STARS", "3.5"))

print(f"Risk label definition:")
print(f"  - Quality decline ≥ {DECLINE_THRESHOLD} stars (historical avg - future avg)")
print(f"  - Future average ≤ {FUTURE_MAX_AVG_STARS} stars")

### Cross-Validation and Viability Constraints

To perform meaningful cross-validation with imbalanced classes, we need:
- **MIN_POSITIVE_EXAMPLES = 25** (real mode): At least 25 risk=1 businesses across the entire dataset
- **MIN_CLASS_RATE = 10%** (real mode): The minority class must be at least 10% of the data
- **MIN_EXAMPLES_PER_CV_FOLD = 5** (real mode): Each fold must have at least 5 examples per class

**What happens if these aren't met?** The notebook automatically tries relaxed thresholds (e.g., decline ≥ 0.50 instead of 0.75) or raises an error guiding students to adjust MAX_BUSINESSES.

In [ ]:
CV_SPLITS = int(os.environ.get("TASK6_CV_SPLITS", "3"))
DEFAULT_MIN_POSITIVE_EXAMPLES = 4 if not REAL_DATA_MODE else 25
DEFAULT_MIN_CLASS_RATE = 0.05 if not REAL_DATA_MODE else 0.10
DEFAULT_MIN_EXAMPLES_PER_CV_FOLD = 2 if not REAL_DATA_MODE else 5

MIN_POSITIVE_EXAMPLES = int(os.environ.get("TASK6_MIN_POSITIVE_EXAMPLES", str(DEFAULT_MIN_POSITIVE_EXAMPLES)))
MIN_CLASS_RATE = float(os.environ.get("TASK6_MIN_CLASS_RATE", str(DEFAULT_MIN_CLASS_RATE)))
MIN_EXAMPLES_PER_CV_FOLD = int(os.environ.get("TASK6_MIN_EXAMPLES_PER_CV_FOLD", str(DEFAULT_MIN_EXAMPLES_PER_CV_FOLD)))
MAX_TEXT_CHARS_PER_BUSINESS = int(os.environ.get("TASK6_MAX_TEXT_CHARS_PER_BUSINESS", "12000"))

print(f"Cross-validation: {CV_SPLITS} folds")
print(f"Viability requirements:")
print(f"  - At least {MIN_POSITIVE_EXAMPLES} positive examples")
print(f"  - Minority class rate ≥ {MIN_CLASS_RATE:.0%}")
print(f"  - At least {MIN_EXAMPLES_PER_CV_FOLD} examples per class per fold")

In [ ]:
ARTIFACT_DIR = Path(os.environ.get("TASK6_ARTIFACT_DIR", os.environ.get("DMC_ARTIFACT_DIR", "artifacts/task6")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifact directory configured:")
print(f"  - Task 6 output: {ARTIFACT_DIR.resolve()}")
print("Configuration complete. ✓")


### Helper Functions: The Scaffolding

The functions below handle three concerns:
1. **I/O helpers** (`save_table`, `save_json`, `save_figure`): Consistent artifact saving with size feedback
2. **Category processing** (`normalize_categories`, `is_restaurant_or_food`, `category_matches`): Handle Yelp's messy category strings
3. **Yelp dataset utilities** (`find_file`, `extract_nested_zips`): Navigate Kaggle's nested ZIP structure
4. **Feature engineering** (`simple_sentiment`, `join_limited_text`): Transform review text into numeric signals

**Key insight**: These helpers are reusable across tasks. In a production system, they'd live in a shared utilities module.

In [ ]:
def save_table(df: pd.DataFrame, name: str, index: bool = False) -> Path:
    """Save DataFrame to CSV with feedback."""
    path = ARTIFACT_DIR / name
    df.to_csv(path, index=index)
    print(f"Saved {path} ({len(df):,} rows)")
    return path


def save_json(obj, name: str) -> Path:
    """Save Python object to JSON with pretty formatting."""
    path = ARTIFACT_DIR / name
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    print("Saved", path)
    return path


def save_figure(name: str) -> Path:
    """Save current matplotlib figure with high DPI."""
    path = ARTIFACT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved", path)
    return path


print("I/O helpers defined.")

In [ ]:
def normalize_categories(categories) -> list[str]:
    """Parse Yelp category field into list of strings."""
    if categories is None or (isinstance(categories, float) and pd.isna(categories)):
        return []
    if isinstance(categories, list):
        return [str(c).strip() for c in categories if str(c).strip()]
    return [c.strip() for c in str(categories).split(",") if c.strip()]


def categories_text(categories) -> str:
    """Convert category list to readable pipe-separated string."""
    return " | ".join(normalize_categories(categories))


def is_restaurant_or_food(categories) -> bool:
    """Check if business has 'Restaurants' or 'Food' category."""
    cats = {c.lower() for c in normalize_categories(categories)}
    return bool({"restaurants", "food"} & cats) if cats else False


def category_matches(categories, target: str) -> bool:
    """Check if any category exactly matches target (case-insensitive)."""
    if not target:
        return True
    target_l = target.lower()
    return any(c.lower() == target_l for c in normalize_categories(categories))


print("Category processing helpers defined.")

In [ ]:
def find_file(root, patterns):
    """Find first file matching any glob pattern under root."""
    root = Path(root)
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None


def extract_nested_zips(root):
    """Extract any ZIP files found under root (Kaggle nests ZIPs)."""
    root = Path(root)
    for zip_path in root.rglob("*.zip"):
        extract_dir = zip_path.with_suffix("")
        marker = extract_dir / ".extracted"
        if marker.exists():
            continue
        extract_dir.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)
            marker.write_text("ok", encoding="utf-8")
            print("Extracted", zip_path, "to", extract_dir)
        except Exception as exc:
            print("Could not extract", zip_path, ":", exc)


def join_limited_text(texts, max_chars=MAX_TEXT_CHARS_PER_BUSINESS):
    """Concatenate review texts up to character limit."""
    pieces = []
    total = 0
    for text in texts:
        clean = re.sub(r"\s+", " ", str(text)).strip()
        if not clean:
            continue
        if total + len(clean) + 1 > max_chars:
            remaining = max_chars - total
            if remaining > 200:
                pieces.append(clean[:remaining])
            break
        pieces.append(clean)
        total += len(clean) + 1
    return " ".join(pieces)


print("Dataset utilities defined.")

#### Simple Lexicon-Based Sentiment

This sentiment function is intentionally **simple** — it counts positive vs negative keywords. It's not state-of-the-art, but it's:
- **Fast**: No model loading
- **Transparent**: Students can inspect the word lists
- **Good enough**: For historical trend features, we don't need transformer-level precision

**Pedagogical point**: Not every feature needs a neural network. Sometimes a 20-word lexicon gives you 80% of the signal at 1% of the cost.

In [ ]:
# TODO (Category B — Classic data mining): Implement simple lexicon-based sentence sentiment.
# Estimated time: ~1.5 h
#
# simple_sentiment(text) → float in [-1, 1]
#   This function is called inside aggregate_business_windows() to compute a sentiment
#   feature for each review BEFORE the cutoff date (historical window).
#
# Implementation:
#   1. Tokenize text with re.findall(r"[a-z]+", text.lower()).
#   2. Count positive and negative tokens against small word sets (define them here).
#   3. Return (pos_count - neg_count) / max(1, total_tokens), clamped to [-1, 1].
#   4. Return 0.0 for empty / non-string input.
#
# Keep the word sets small (10–16 words each) — this is a simple feature, not a fine-tuned model.
# The goal is a signal that correlates with restaurant quality decline.
#
# Note: this sentiment score becomes the "historical_sentiment" feature in the prediction frame.
#       Choose words carefully: what language in reviews predicts future quality decline?

_POSITIVE_TOKENS = {
    # TODO: 10–16 positive food/service words
}
_NEGATIVE_TOKENS = {
    # TODO: 10–16 negative food/service words
}

def simple_sentiment(text: str) -> float:
    # TODO: tokenize; count positive/negative; clamp to [-1, 1]; handle edge cases
    pass  # ← replace

# --- Validation ---
# assert simple_sentiment("excellent great delicious fresh") > 0
# assert simple_sentiment("terrible bland cold disappointing") < 0
# assert simple_sentiment("") == 0.0
# print("\u2713 simple_sentiment ready")


# Xiaofei's Task 6 TODO implementation / 中文注释：实现轻量级餐厅评论情感词典。
# 中文注释：覆盖原空集合，加入餐厅评论中常见的正向词。
_POSITIVE_TOKENS = {"amazing", "awesome", "clean", "crispy", "delicious", "excellent", "favorite", "flavorful", "fresh", "friendly", "great", "juicy", "perfect", "tasty", "tender", "wonderful"}
# 中文注释：覆盖原空集合，加入餐厅评论中常见的负向词。
_NEGATIVE_TOKENS = {"awful", "bad", "bland", "cold", "dirty", "disappointing", "dry", "greasy", "mediocre", "overcooked", "overpriced", "rude", "slow", "stale", "terrible", "worst"}

# 中文注释：重新定义简单情感函数，覆盖上方 pass 占位实现。
def simple_sentiment(text: str) -> float:
    # 中文注释：空文本或非字符串输入按中性情感处理。
    if not isinstance(text, str) or not text.strip():
        # 中文注释：返回中性分数。
        return 0.0
    # 中文注释：使用题目指定的正则从小写文本中提取英文 token。
    tokens = re.findall(r"[a-z]+", text.lower())
    # 中文注释：没有可用 token 时按中性情感处理。
    if not tokens:
        # 中文注释：返回中性分数。
        return 0.0
    # 中文注释：统计正向词典命中的 token 数。
    pos_count = sum(token in _POSITIVE_TOKENS for token in tokens)
    # 中文注释：统计负向词典命中的 token 数。
    neg_count = sum(token in _NEGATIVE_TOKENS for token in tokens)
    # 中文注释：按 token 总数归一化正负词命中差。
    score = (pos_count - neg_count) / max(1, len(tokens))
    # 中文注释：将分数裁剪到 [-1, 1] 区间并转成普通浮点数。
    return float(np.clip(score, -1.0, 1.0))

# 中文注释：运行正向样例断言，确认正向词会产生正分。
assert simple_sentiment("excellent great delicious fresh") > 0
# 中文注释：运行负向样例断言，确认负向词会产生负分。
assert simple_sentiment("terrible bland cold disappointing") < 0
# 中文注释：运行空文本断言，确认缺失文本按中性处理。
assert simple_sentiment("") == 0.0
# 中文注释：打印情感函数准备完成的信息。
print("✓ simple_sentiment ready")


## 2. Load Yelp-Shaped Data

The required dataset is Yelp review/business data. Smoke mode creates synthetic restaurants with clear historical and future windows. Real modes stream bounded subsets of Yelp JSON lines rather than loading the full review file into memory.

### Smoke Data: Controlled Synthetic Test Cases

For smoke-test validation and student local testing, we generate **synthetic Yelp-shaped data** with known risk patterns:

- **72 businesses** across 6 cuisines (Chinese, Mexican, Japanese, Indian, Italian, Thai)
- **Historical window**: 7-13 reviews per business, 2019-2020 dates, stars drawn from cuisine-specific distributions
- **Future window**: 5-10 reviews per business, 2021-2023 dates
- **Risk pattern**: Every 4th business (i % 4 == 0) gets quality decline (historical mean 3.8-4.7 → future mean 2.0-3.2)

**Why synthetic data matters**: Students can verify that their label logic works *before* debugging on real messy Yelp data. The smoke test should produce ~18 risk=1 businesses and complete in under 30 seconds.

## About Truncated SVD (LSA)

Truncated SVD projects high-dimensional sparse TF-IDF text features into a smaller dense representation, often called latent semantic analysis when applied to text. In Task 6, it is used as **M4_LSA_TextPlusMeta**, a required model in the comparison: does compressing sparse TF-IDF text into lower-dimensional latent semantic features help or hurt leakage-safe future risk prediction?

LSA can reduce dimensionality and capture broad co-occurrence structure, but it may also make feature interpretation less direct and discard rare but informative terms. Evaluate it with the same stratified cross-validation and macro-F1 criteria as the simpler baselines rather than assuming dimensionality reduction improves performance.

**Key Resources:**
- [scikit-learn TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) - Official API.
- [scikit-learn decomposition guide](https://scikit-learn.org/stable/modules/decomposition.html#lsa) - LSA/SVD documentation.
- [Wikipedia: Latent semantic analysis](https://en.wikipedia.org/wiki/Latent_semantic_analysis) - Conceptual overview.

In [ ]:
def make_smoke_yelp_prediction_data(seed=RANDOM_SEED):
    """Generate synthetic Yelp businesses and reviews with controlled quality-decline patterns."""
    rng = random.Random(seed)
    cuisine_templates = [
        ("Chinese", ["dumplings", "noodles", "fried rice"]),
        ("Mexican", ["tacos", "burritos", "salsa"]),
        ("Japanese", ["sushi", "ramen", "tempura"]),
        ("Indian", ["biryani", "naan", "curry"]),
        ("Italian", ["pizza", "pasta", "risotto"]),
        ("Thai", ["pad thai", "curry", "noodles"]),
    ]
    businesses = []
    reviews = []
    rid = 0
    
    for i in range(72):
        cuisine, dishes = cuisine_templates[i % len(cuisine_templates)]
        city = ["Smokeville", "Teston", "Example City"][i % 3]
        state = "CA" if i % 4 else "IL"
        business_id = f"biz_{i:03d}"
        risk_pattern = i % 4 == 0 or (i % 11 == 0)  # ~25% will be at risk
        
        businesses.append({
            "business_id": business_id,
            "name": f"{cuisine} Place {i:02d}",
            "categories": f"Restaurants, {cuisine}",
            "city": city,
            "state": state,
            "is_open": 0 if risk_pattern and i % 8 == 0 else 1,
        })
        
        # Historical reviews (before cutoff)
        n_hist = rng.randint(7, 13)
        hist_mean = rng.uniform(3.8, 4.7) if risk_pattern else rng.uniform(3.0, 4.3)
        for j in range(n_hist):
            dish = rng.choice(dishes)
            stars = int(np.clip(round(rng.gauss(hist_mean, 0.55)), 1, 5))
            tone = "great fresh friendly delicious" if stars >= 4 else "mixed okay slow"
            text = f"Historical visit: the {dish} was {tone}. Service and atmosphere were part of the experience."
            date = pd.Timestamp("2019-01-01") + pd.Timedelta(days=rng.randint(0, 720))
            reviews.append({
                "review_id": f"r_{rid}",
                "business_id": business_id,
                "stars": stars,
                "date": date.strftime("%Y-%m-%d"),
                "text": text
            })
            rid += 1
        
        # Future reviews (after cutoff)
        n_future = rng.randint(5, 10)
        future_mean = rng.uniform(2.0, 3.2) if risk_pattern else min(5.0, hist_mean + rng.uniform(-0.25, 0.55))
        for j in range(n_future):
            dish = rng.choice(dishes)
            stars = int(np.clip(round(rng.gauss(future_mean, 0.6)), 1, 5))
            tone = "cold slow disappointing stale worse" if risk_pattern else "fresh friendly delicious consistent"
            text = f"Future visit: the {dish} was {tone}. The recent experience shaped my rating."
            date = pd.Timestamp("2021-02-01") + pd.Timedelta(days=rng.randint(0, 850))
            reviews.append({
                "review_id": f"r_{rid}",
                "business_id": business_id,
                "stars": stars,
                "date": date.strftime("%Y-%m-%d"),
                "text": text
            })
            rid += 1
    
    return pd.DataFrame(businesses), pd.DataFrame(reviews)


print("Smoke data generator defined.")

### Real Data Loading: Bounded Streaming

**Why not load all 7M Yelp reviews into memory?** It's unnecessary and slow. Instead, we:
1. Stream business JSON line-by-line, stopping at MAX_BUSINESSES (2,000 by default)
2. Filter to restaurant/food businesses matching TARGET_CATEGORY if specified
3. Stream review JSON, keeping only reviews for our selected businesses, stopping at MAX_REVIEWS_TOTAL (200K)

**Key insight**: This "streaming subset" approach is how you handle large datasets in production — load only what you need, when you need it.

In [ ]:
def load_bounded_yelp_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    if DATASET_MODE == "smoke_test":
        business_df, review_df = make_smoke_yelp_prediction_data()
        business_df = business_df.copy()
        review_df = review_df.copy()
        review_df["date"] = pd.to_datetime(review_df["date"], errors="coerce")
        return business_df, review_df

    data_root = DATA_ROOT
    if DATASET_MODE == "kaggle_latest":
        try:
            import kagglehub
            data_root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        except Exception as exc:
            warnings.warn(f"kagglehub download failed ({exc}); falling back to DATA_ROOT={DATA_ROOT}")
            data_root = DATA_ROOT

    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"Task 6 data root not found: {data_root}")

    extract_nested_zips(data_root)
    business_path = find_file(data_root, ["yelp_academic_dataset_business.json", "**/yelp_academic_dataset_business.json"])
    review_path = find_file(data_root, ["yelp_academic_dataset_review.json", "**/yelp_academic_dataset_review.json"])
    if business_path is None or review_path is None:
        raise FileNotFoundError(
            f"Could not locate Yelp business/review JSON under {data_root}. "
            f"business={business_path}, review={review_path}"
        )

    business_rows = []
    selected_business_ids = set()
    with Path(business_path).open("r", encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            categories = record.get("categories")
            if REQUIRE_RESTAURANTS and not is_restaurant_or_food(categories):
                continue
            if TARGET_CATEGORY and not category_matches(categories, TARGET_CATEGORY):
                continue
            business_rows.append({
                "business_id": record.get("business_id"),
                "name": record.get("name", "unknown"),
                "categories": record.get("categories", ""),
                "city": record.get("city", "unknown"),
                "state": record.get("state", "unknown"),
                "is_open": record.get("is_open", np.nan),
            })
            selected_business_ids.add(record.get("business_id"))
            if len(business_rows) >= MAX_BUSINESSES:
                break

    if not business_rows:
        raise RuntimeError(
            "No businesses matched the Task 6 filters. Adjust TASK6_TARGET_CATEGORY or data source settings."
        )

    review_rows = []
    per_business_counts = Counter()
    with Path(review_path).open("r", encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            business_id = record.get("business_id")
            if business_id not in selected_business_ids:
                continue
            if per_business_counts[business_id] >= MAX_REVIEWS_PER_BUSINESS:
                continue
            review_rows.append({
                "review_id": record.get("review_id"),
                "business_id": business_id,
                "stars": record.get("stars"),
                "date": record.get("date"),
                "text": record.get("text", ""),
            })
            per_business_counts[business_id] += 1
            if len(review_rows) >= MAX_REVIEWS_TOTAL:
                break

    business_df = pd.DataFrame(business_rows)
    review_df = pd.DataFrame(review_rows)
    if review_df.empty:
        raise RuntimeError("No reviews matched the selected Task 6 businesses within the configured bounds.")
    review_df["date"] = pd.to_datetime(review_df["date"], errors="coerce")
    review_df = review_df.dropna(subset=["date"]).copy()
    return business_df, review_df


business_df, review_df = load_bounded_yelp_data()
print(f"Loaded {len(business_df):,} businesses and {len(review_df):,} reviews for Task 6.")
display(business_df.head(5))
display(review_df.head(5))

## 3. Build the Future Quality-Decline Prediction Frame

This is the **heart of Task 6**: converting Yelp reviews into a supervised learning problem.

**The three-step process**:
1. **Split reviews by cutoff date** → historical window (features) vs future window (labels)
2. **Aggregate by business** → historical average stars, historical review counts, historical sentiment, and historical text
3. **Compute labels** → risk=1 if future quality declines enough and the future average is low enough

**Critical constraints**:
- Only businesses with enough historical and future reviews survive.
- If the resulting class balance is unviable, the notebook adjusts thresholds or errors with guidance.

**What you should learn**:
- Target definition is **not arbitrary** — it must be statistically learnable.
- Review support affects label reliability.
- The prediction frame is where leakage-safe supervised data mining is either made or broken.


In [ ]:
def aggregate_business_windows(review_df: pd.DataFrame, business_df: pd.DataFrame) -> pd.DataFrame:
    hist = review_df[review_df["date"] < CUTOFF_DATE].copy()
    fut = review_df[review_df["date"] >= CUTOFF_DATE].copy()
    hist["sentiment"] = hist["text"].map(simple_sentiment)
    fut["sentiment"] = fut["text"].map(simple_sentiment)

    hist_agg = hist.groupby("business_id").agg(
        historical_review_count=("review_id", "count"),
        historical_avg_stars=("stars", "mean"),
        historical_sentiment=("sentiment", "mean"),
        historical_low_star_share=("stars", lambda s: float((s <= 2).mean())),
        historical_text=("text", lambda s: join_limited_text(s, MAX_TEXT_CHARS_PER_BUSINESS)),
        max_historical_review_date=("date", "max"),
    ).reset_index()
    hist_agg["historical_text_length"] = hist_agg["historical_text"].str.len().fillna(0)

    fut_agg = fut.groupby("business_id").agg(
        future_review_count=("review_id", "count"),
        future_avg_stars=("stars", "mean"),
        future_sentiment=("sentiment", "mean"),
        min_future_review_date=("date", "min"),
    ).reset_index()

    frame = business_df.merge(hist_agg, on="business_id", how="left").merge(fut_agg, on="business_id", how="left")
    frame["historical_review_count"] = frame["historical_review_count"].fillna(0).astype(int)
    frame["future_review_count"] = frame["future_review_count"].fillna(0).astype(int)
    frame = frame[(frame["historical_review_count"] >= MIN_HISTORICAL_REVIEWS) & (frame["future_review_count"] >= MIN_FUTURE_REVIEWS)].copy()
    if frame.empty:
        raise RuntimeError("No businesses have enough historical and future reviews. Adjust cutoff date or minimum review thresholds.")
    return frame


def assign_decline_label(frame: pd.DataFrame, threshold: float, future_max: float) -> tuple[pd.DataFrame, dict]:
    candidates = [(threshold, future_max), (0.50, max(future_max, 3.75)), (0.25, max(future_max, 4.00)), (0.10, 5.00)]
    chosen = None
    labeled = None
    viability_rows = []
    for th, mx in candidates:
        temp = frame.copy()
        temp["quality_decline"] = temp["historical_avg_stars"] - temp["future_avg_stars"]
        temp["risk_label"] = ((temp["quality_decline"] >= th) & (temp["future_avg_stars"] <= mx)).astype(int)
        counts = temp["risk_label"].value_counts().sort_index().to_dict()
        positives = int(counts.get(1, 0))
        negatives = int(counts.get(0, 0))
        total = positives + negatives
        min_class_count = min(positives, negatives) if total else 0
        minority_rate = min_class_count / total if total else 0.0
        viable = (
            positives >= MIN_POSITIVE_EXAMPLES
            and negatives >= MIN_POSITIVE_EXAMPLES
            and minority_rate >= MIN_CLASS_RATE
        )
        viability_rows.append({
            "decline_threshold": th,
            "future_max_avg_stars": mx,
            "eligible_businesses": total,
            "non_risk_count": negatives,
            "risk_count": positives,
            "risk_rate": positives / total if total else 0.0,
            "minority_class_rate": minority_rate,
            "meets_min_positive_examples": positives >= MIN_POSITIVE_EXAMPLES,
            "meets_min_class_rate": minority_rate >= MIN_CLASS_RATE,
            "viable_for_modeling": viable,
        })
        if viable:
            chosen = {
                "decline_threshold": th,
                "future_max_avg_stars": mx,
                "threshold_tuned_for_class_balance": (th != threshold or mx != future_max),
            }
            labeled = temp
            break
    if labeled is None:
        summary = pd.DataFrame(viability_rows)
        best = summary.sort_values(["viable_for_modeling", "risk_count", "minority_class_rate"], ascending=False).head(1)
        detail = best.to_dict("records")[0] if not best.empty else {}
        raise RuntimeError(
            "Future-risk target is not viable for real modeling under the current data bounds/thresholds. "
            f"Need at least {MIN_POSITIVE_EXAMPLES} examples in each class and minority-class rate >= {MIN_CLASS_RATE:.0%}. "
            f"Best candidate observed: {detail}. "
            "Concrete fixes: increase TASK6_MAX_BUSINESSES and TASK6_MAX_REVIEWS_TOTAL, use a broader category, adjust "
            "TASK6_CUTOFF_DATE, lower TASK6_MIN_FUTURE_REVIEWS cautiously, or relax TASK6_DECLINE_THRESHOLD / "
            "TASK6_FUTURE_MAX_AVG_STARS while documenting the target change."
        )
    chosen["class_counts"] = {str(k): int(v) for k, v in labeled["risk_label"].value_counts().sort_index().to_dict().items()}
    chosen["minimum_viability_requirements"] = {
        "min_positive_examples": MIN_POSITIVE_EXAMPLES,
        "min_class_rate": MIN_CLASS_RATE,
        "min_examples_per_cv_fold": MIN_EXAMPLES_PER_CV_FOLD,
    }
    chosen["candidate_label_counts"] = viability_rows
    return labeled, chosen


def add_historical_slopes(frame: pd.DataFrame, reviews: pd.DataFrame) -> pd.DataFrame:
    hist = reviews[reviews["date"] < CUTOFF_DATE].copy()
    hist["time_bin"] = hist["date"].dt.to_period("Q").astype(str)
    rows = []
    for bid, group in hist.groupby("business_id"):
        by_time = group.groupby("time_bin").agg(review_count=("review_id", "count"), avg_stars=("stars", "mean")).sort_index()
        rows.append({"business_id": bid, "historical_review_count_slope": _slope_for_array(by_time["review_count"]), "historical_stars_slope": _slope_for_array(by_time["avg_stars"])})
    return frame.merge(pd.DataFrame(rows), on="business_id", how="left")


def _slope_for_array(values) -> float:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(arr) < 2:
        return 0.0
    return float(np.polyfit(np.arange(len(arr), dtype=float), arr, 1)[0])

prediction_frame = aggregate_business_windows(review_df, business_df)
prediction_frame, label_config = assign_decline_label(prediction_frame, DECLINE_THRESHOLD, FUTURE_MAX_AVG_STARS)
prediction_frame = add_historical_slopes(prediction_frame, review_df)

prediction_frame["primary_category"] = prediction_frame["categories"].map(lambda x: next((c for c in normalize_categories(x) if c.lower() not in {"restaurants", "food"}), "unknown"))
prediction_frame["state"] = prediction_frame["state"].fillna("unknown")
prediction_frame["city"] = prediction_frame["city"].fillna("unknown")
prediction_frame["categories_text"] = prediction_frame["categories"].map(categories_text)

prediction_frame["future_review_support_tier"] = pd.cut(
    prediction_frame["future_review_count"],
    bins=[-np.inf, 6, 10, 20, np.inf],
    labels=["minimum_to_6", "7_to_10", "11_to_20", "21_plus"],
)

class_counts = prediction_frame["risk_label"].value_counts().sort_index().rename_axis("risk_label").reset_index(name="count")
target_viability_summary = pd.DataFrame(label_config.get("candidate_label_counts", []))
future_label_support = (
    prediction_frame.groupby("future_review_support_tier", observed=True)
    .agg(
        businesses=("business_id", "count"),
        risk_count=("risk_label", "sum"),
        risk_rate=("risk_label", "mean"),
        median_future_reviews=("future_review_count", "median"),
        median_quality_decline=("quality_decline", "median"),
    )
    .reset_index()
)
save_table(class_counts, "class_balance.csv")
save_table(target_viability_summary, "target_threshold_sensitivity.csv")
save_table(future_label_support, "future_label_support_by_review_count.csv")
print("Label config:", {k: v for k, v in label_config.items() if k != "candidate_label_counts"})
display(class_counts)
display(target_viability_summary)
display(future_label_support)
display(prediction_frame[["business_id", "name", "historical_review_count", "future_review_count", "future_review_support_tier", "historical_avg_stars", "future_avg_stars", "quality_decline", "risk_label"]].head(20))

feature_frame = prediction_frame.copy()
y = feature_frame["risk_label"].astype(int)
y_all = y
X_all = feature_frame.copy()
save_table(feature_frame, "business_prediction_frame.csv")
print("Saved business_prediction_frame.csv and prepared feature_frame for modeling.")


## 5. Feature Groups and Model Pipelines

The required model comparison uses three model families:

| Model | Purpose | Features |
|---|---|---|
| **M1 — Metadata baseline** | Structured numeric signal | Historical count, stars, sentiment |
| **M2 — Text only** | Text-only TF-IDF signal | Historical review text |
| **M3 — Text + metadata** | Combined model | TF-IDF + numeric metadata |
| **M4 — LSA + metadata** | Compressed latent text | TruncatedSVD(TF-IDF) + numeric metadata |

**TF-IDF vs. LSA/SVD**: TF-IDF keeps a high-dimensional sparse vocabulary representation — one coefficient per term. LSA/SVD compresses TF-IDF into fewer latent semantic dimensions, potentially reducing noise and capturing co-occurrence structure at the cost of term-level interpretability. Comparing M3 (sparse TF-IDF) and M4 (compressed LSA) under the same leakage-safe framework reveals whether dimensionality reduction helps or hurts for this prediction task.

This keeps Task 6 focused on the core data-mining question: do historical structured signals, text signals, or their combination predict future restaurant risk?


## About TF-IDF Vectorization

TF-IDF vectorization converts text into weighted term features that increase the importance of terms that are common within a document but relatively uncommon across the corpus. In Task 6, it provides the core sparse text features for supervised quality-decline prediction, converting aggregated historical review text into weighted term vectors that the classifier uses to identify vocabulary patterns predictive of future risk. This is a direct application of representation choice in a supervised context: the same review text can produce different prediction results depending on whether raw counts, TF-IDF weights, or character n-grams are used.

You use TF-IDF here to build the text-only classifier pipeline and as the text component of the combined text+metadata model. Because TF-IDF features remain sparse and inspectable, they also support interpretation: examining the top positive and negative coefficients from a logistic regression trained on TF-IDF features reveals which review vocabulary signals quality decline. Compared with sentence-transformer embeddings, TF-IDF is less semantically rich but often more stable on small labeled samples and more straightforward to interpret in a prediction context.

**Key Resources:**
- [scikit-learn TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) - Official API for TF-IDF feature extraction.
- [scikit-learn Text Feature Extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) - Guide to count and TF-IDF representations.
- [Wikipedia: TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) - Conceptual overview of term-frequency and inverse-document-frequency weighting.

Why use this method?
- Produces interpretable prediction features directly linked to review vocabulary.
- Offers a strong classical text baseline for the supervised comparison table.
- Supports coefficient-level model interpretation for the feature importance section.

## About Logistic Regression (TF-IDF features)

Logistic regression is a linear supervised classifier that works well with sparse TF-IDF review-text features. In Task 6, it provides an interpretable baseline for predicting future restaurant risk from historical text and metadata. Students can compare its macro-F1 against dummy, metadata-only, and advanced models while inspecting coefficients for predictive signals.

The model is not automatically causal or policy-ready. Coefficients show associations in the training data, and small labeled samples can make estimates unstable. Still, logistic regression is a strong capstone endpoint because it links text mining features to disciplined supervised evaluation.

**Key Resources:**
- [scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) - Official API.
- [scikit-learn text classification tutorial](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) - Text classification workflow.
- [Wikipedia: Logistic regression](https://en.wikipedia.org/wiki/Logistic_regression) - Conceptual overview.

## About Stratified K-Fold Cross-Validation

Stratified K-fold cross-validation splits labeled examples into folds while preserving class proportions as much as possible. In Task 6, this is essential because risk/pass labels can be imbalanced and the labeled sample is small. Stratification reduces the chance that a fold has too few positive examples to evaluate meaningfully.

Students use the same folds across model comparisons when feasible so differences reflect model behavior rather than accidental split variation. Cross-validation is still an estimate, not a guarantee of future deployment performance, so error analysis and error analysis remain necessary.

**Key Resources:**
- [scikit-learn StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) - Official API.
- [scikit-learn cross-validation guide](https://scikit-learn.org/stable/modules/cross_validation.html) - Evaluation documentation.
- [Wikipedia: Cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) - Conceptual overview.

In [ ]:
# TODO (Category C — Method design): Design the feature groups and model pipelines.
# Estimated time: ~1.5 h
#
# Task 6 requires testing multiple classifiers. Define THREE or more pipelines below.
# Use sklearn Pipeline to chain preprocessing and a classifier.
#
# Required model types (see learning objectives):
#   1. Baseline: DummyClassifier (strategy="most_frequent") — no features needed.
#   2. Metadata only: numeric features (base_numeric_features) + optional categorical.
#      Use SimpleImputer(strategy="median") + StandardScaler + LogisticRegression.
#   3. Text only: TF-IDF on "historical_text" + LogisticRegression or LinearSVC.
#   4. Text + metadata: ColumnTransformer combining TF-IDF text and numeric features.
#   5. Optional advanced: TruncatedSVD on TF-IDF features, or class_weight="balanced".
#
# For each pipeline, you will call:
#   results, oof_preds = evaluate_model(name, pipeline, feature_frame, y_all)
# where feature_frame is the full prediction DataFrame (the evaluate_model function
# selects its own features from it using the pipeline's ColumnTransformer).
#
# Guidance on class imbalance (see the markdown cell above):
#   - Add class_weight="balanced" to LogisticRegression when the minority class is rare.
#   - DummyClassifier establishes the macro-F1 floor.

from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC

base_numeric_features = [
    "historical_review_count",
    "historical_avg_stars",
    "historical_sentiment",
    # TODO: add more features from feature_frame if useful
    #   e.g., "historical_low_star_share", "future_review_count", "city_encoded"
    #   WARNING: do NOT include future_* columns — that is leakage!
]

# TODO: define categorical_features (e.g., ["primary_category", "state"])
categorical_features = []  # replace

# TODO: build model_pipelines — a list of (name_str, sklearn_pipeline) tuples
model_pipelines = [
    # ("Dummy baseline",     Pipeline([("clf", DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED))])),
    # ("Numeric LR",         Pipeline([...])),
    # ("TF-IDF LR",          Pipeline([...])),
    # ("TF-IDF + meta LR",   Pipeline([...])),
]



# Xiaofei's Task 6 TODO implementation / 中文注释：定义无时间泄漏的特征组与模型管线。
# 中文注释：导入列级预处理器，用来把文本、数值和类别特征组合到同一个模型里。
from sklearn.compose import ColumnTransformer
# 中文注释：导入缺失值填补器，保证真实数据中的缺失数值和类别不会中断训练。
from sklearn.impute import SimpleImputer
# 中文注释：导入逻辑回归分类器，作为可解释的线性监督模型。
from sklearn.linear_model import LogisticRegression
# 中文注释：导入 Pipeline，串联预处理和分类器。
from sklearn.pipeline import Pipeline
# 中文注释：导入标准化器，让数值特征适合逻辑回归优化。
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# 中文注释：导入 TF-IDF 向量器，把历史评论文本转换为稀疏词项特征。
from sklearn.feature_extraction.text import TfidfVectorizer
# 中文注释：导入 TruncatedSVD，构建 LSA 压缩文本表示模型。
from sklearn.decomposition import TruncatedSVD

# 中文注释：在原始安全数值特征之外加入历史窗口内可得的低星占比、文本长度和时间斜率。
base_numeric_features = list(dict.fromkeys(base_numeric_features + ["historical_low_star_share", "historical_text_length", "historical_review_count_slope", "historical_stars_slope"]))
# 中文注释：过滤掉当前特征表中不存在的列，避免不同数据版本导致 KeyError。
base_numeric_features = [col for col in base_numeric_features if col in feature_frame.columns]
# 中文注释：使用菜系和州作为类别特征，这些都来自商家元数据而不是 future label。
categorical_features = [col for col in ["primary_category", "state"] if col in feature_frame.columns]
# 中文注释：构建数值特征预处理流程：中位数填补后标准化。
numeric_preprocess = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
# 中文注释：优先使用新版 sklearn 的 sparse_output 参数创建 one-hot 编码器。
try:
    # 中文注释：新版 sklearn 使用 sparse_output 控制稀疏输出。
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
# 中文注释：兼容旧版 sklearn 中尚未支持 sparse_output 的情况。
except TypeError:
    # 中文注释：旧版 sklearn 使用 sparse 参数控制稀疏输出。
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)
# 中文注释：构建类别特征预处理流程：众数填补后 one-hot 编码。
categorical_preprocess = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", one_hot_encoder)])
# 中文注释：构建元数据模型的列转换器，只包含历史数值和安全类别特征。
metadata_preprocess = ColumnTransformer([("numeric", numeric_preprocess, base_numeric_features), ("categorical", categorical_preprocess, categorical_features)], remainder="drop")
# 中文注释：构建文本-only 模型的列转换器，只读取 historical_text。
text_preprocess = ColumnTransformer([("text", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True), "historical_text")], remainder="drop")
# 中文注释：构建文本加元数据模型的列转换器，组合历史文本、历史数值和安全类别特征。
text_meta_preprocess = ColumnTransformer([("text", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True), "historical_text"), ("numeric", numeric_preprocess, base_numeric_features), ("categorical", categorical_preprocess, categorical_features)], remainder="drop")
# 中文注释：根据样本量选择 LSA 维度，避免维度超过训练样本数。
lsa_components = max(2, min(64, len(feature_frame) - 1))
# 中文注释：构建 LSA 文本预处理流程：TF-IDF 后用 SVD 压缩。
lsa_text_pipeline = Pipeline([("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True)), ("svd", TruncatedSVD(n_components=lsa_components, random_state=RANDOM_SEED))])
# 中文注释：构建 LSA 加元数据模型的列转换器。
lsa_meta_preprocess = ColumnTransformer([("lsa_text", lsa_text_pipeline, "historical_text"), ("numeric", numeric_preprocess, base_numeric_features), ("categorical", categorical_preprocess, categorical_features)], remainder="drop")
# 中文注释：重新初始化模型管线列表，覆盖上方占位空列表。
model_pipelines = []
# 中文注释：加入多数类 Dummy 基线，建立宏 F1 的最低参照。
model_pipelines.append(("Dummy baseline", Pipeline([("clf", DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED))])))
# 中文注释：加入元数据-only 逻辑回归模型，检验不用文本时的预测能力。
model_pipelines.append(("Metadata LR", Pipeline([("ct", metadata_preprocess), ("clf", LogisticRegression(class_weight="balanced", C=0.5, max_iter=1000, random_state=RANDOM_SEED))])))
# 中文注释：加入 TF-IDF 文本-only 逻辑回归模型，检验历史评论词项信号。
model_pipelines.append(("TF-IDF LR", Pipeline([("ct", text_preprocess), ("clf", LogisticRegression(class_weight="balanced", C=0.5, max_iter=1000, random_state=RANDOM_SEED))])))
# 中文注释：加入 TF-IDF 加元数据逻辑回归模型，检验组合特征是否优于单一来源。
model_pipelines.append(("TF-IDF + Metadata LR", Pipeline([("ct", text_meta_preprocess), ("clf", LogisticRegression(class_weight="balanced", C=0.5, max_iter=1000, random_state=RANDOM_SEED))])))
# 中文注释：加入 LSA 加元数据逻辑回归模型，比较压缩文本表示和稀疏 TF-IDF 的差异。
model_pipelines.append(("LSA + Metadata LR", Pipeline([("ct", lsa_meta_preprocess), ("clf", LogisticRegression(class_weight="balanced", C=0.5, max_iter=1000, random_state=RANDOM_SEED))])))
# 中文注释：打印最终使用的安全数值特征。
print("Numeric features:", base_numeric_features)
# 中文注释：打印最终使用的安全类别特征。
print("Categorical features:", categorical_features)

assert len(model_pipelines) >= 3, "Define at least 3 model pipelines (including baseline)"
print("Model pipelines defined:", [name for name, _ in model_pipelines])


## Data Mining Concept: Class Imbalance as a Core Prediction Challenge

**What is class imbalance?**

In this Task 6 prediction problem, only ~13% of restaurants show future quality decline (risk=1), while ~87% remain stable (risk=0). This severe imbalance creates a fundamental data mining challenge that naive approaches fail to address.

**Why does imbalance break standard evaluation?**

A "model" that always predicts class 0 (non-risk) achieves **87% accuracy** while providing **zero predictive value** — it never identifies a single declining restaurant. This is the classic imbalance trap: accuracy becomes a misleading metric because it conflates majority-class performance with actual predictive skill.

**Three technical responses to class imbalance:**

1. **Macro-F1 as the primary metric**: Instead of accuracy, we use macro-F1, which averages the F1 score for class 0 (non-risk) and class 1 (risk) equally:
   - **Macro-F1 = (F1_class_0 + F1_class_1) / 2**
   - This forces the model to perform well on BOTH classes. A model that predicts all zeros gets F1_class_1 = 0, yielding macro-F1 ≈ 0.43–0.47 (the majority baseline), not 87%.
   - A good model for this problem achieves macro-F1 ≈ 0.60–0.65, meaning it balances non-risk precision/recall (~0.85–0.90 F1) with risk precision/recall (~0.30–0.40 F1).
2. **Stratified K-fold cross-validation**: Each CV fold preserves the same 13% risk rate as the full dataset. Without stratification, some folds might have only 1–2 risk=1 examples, making those folds unlearnable. Stratification ensures every fold is a mini-replica of the full distribution.
   - **StratifiedKFold(n_splits=3, shuffle=True)** — guarantees each fold has ~13% risk=1 businesses
   - **Out-of-fold predictions** (`cross_val_predict`) — trains on folds 1+2, predicts fold 3, rotates through all folds, producing predictions for every business without data leakage
3. **Class weighting in model training**: Most sklearn classifiers support `class_weight="balanced"`, which up-weights the loss for minority-class errors during training. For tree ensembles like balanced classifiers, `scale_pos_weight` serves the same purpose:
   - **scale_pos_weight = (# class_0 examples) / (# class_1 examples) ≈ 6.7**
   - This tells the model: "A false negative (missing a true decline) is ~7× more costly than a false positive (false alarm)."
   - Trade-off: Higher recall (finds more true declines) but lower precision (more false alarms).

**Pedagogical insight for graduate students:**

Class imbalance is not a nuisance to "fix" — it's a fundamental characteristic of many real-world prediction problems (fraud detection, disease diagnosis, equipment failure, churn prediction). The correct response is NOT to resample/SMOTE the data into artificial balance, but to:
- Use evaluation metrics that reflect real-world costs (macro-F1, precision-recall curves, cost-sensitive metrics)
- Design models that explicitly account for class costs (class weights, threshold tuning)
- Interpret results in context: For restaurant risk prediction, is it better to miss real declines (false negatives) or raise false alarms (false positives)? The answer depends on the business use case.

**What you should report:**

- The baseline majority-class performance (all-zeros accuracy vs. macro-F1)
- How class weighting affects precision-recall trade-offs for the minority class
- Whether the model's risk-class F1 (~0.30–0.40) is "good enough" for deployment, given the ~13% base rate
- Whether false positives or false negatives are more costly for a Yelp-native restaurant intelligence system

## 6. Cross-Validated Model Comparison

**Evaluation protocol**:
- **Stratified 3-fold cross-validation** (ensures representative class splits)
- **Macro-F1** is the main metric (treats all classes equally)
- **Per-class precision and recall** for failure analysis
- **Confusion matrix** reveals false-positive vs. false-negative patterns

Use macro-F1 to compare models. Accuracy is misleading for imbalanced classes.

In [ ]:
# TODO (Category B — Classic data mining): Implement stratified CV evaluation + run model comparison.
# Estimated time: ~2 h
#
# ── A) evaluate_model(name, estimator, X, y) → (results_dict, oof_preds_array) ──
#
#   Stratified K-fold CV (use CV_SPLITS, RANDOM_SEED).
#   For each fold: clone estimator, fit on train, predict on test.
#   Metrics: macro-F1, macro-precision, macro-recall.
#   Accumulate out-of-fold predictions in oof_preds (array of length len(y)).
#
#   results_dict keys:
#     name, macro_f1 (mean across folds), macro_precision, macro_recall,
#     std_f1, fold_f1s (list), n_samples, n_positive
#   Return: (results_dict, oof_preds_array)
#
#   Hint: StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)
#
# ── B) Run evaluate_model for every pipeline in model_pipelines ─────────────
#
#   For each (name, estimator) in model_pipelines:
#     res, oof = evaluate_model(name, estimator, feature_frame, y_all)
#   Collect all results into comparison_df (sorted by macro_f1 descending).
#   Save to ARTIFACT_DIR / "model_comparison_results.csv".
#   Display the table.
#
# ── C) Select the best non-baseline model ─────────────────────────────────
#   selected_model_name = comparison_df.iloc[0]["name"]  (or your choice)
#   Store selected_estimator and selected_X for downstream cells.

def evaluate_model(name: str, estimator, X: "pd.DataFrame", y: "pd.Series"):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import f1_score, precision_score, recall_score
    # TODO: implement stratified CV loop with macro-F1 evaluation
    # TODO: return (results_dict, oof_preds_array)
    pass  # ← replace

# TODO: run evaluate_model for each pipeline in model_pipelines
all_results = []
all_oof     = {}

for model_name, estimator in model_pipelines:
    # TODO: call evaluate_model; store results in all_results; store oof in all_oof
    pass  # ← replace

# TODO: build comparison_df and save
comparison_df = None  # replace

# TODO: select best model
selected_model_name = None  # replace
selected_estimator  = None  # replace
selected_X          = feature_frame  # set to appropriate feature subset if needed
y_all               = feature_frame["risk_label"].astype(int) if "risk_label" in feature_frame.columns else None
selected_pred       = all_oof.get(selected_model_name, np.zeros(len(y_all), dtype=int)) if y_all is not None else None

# --- Validation ---
# assert isinstance(comparison_df, pd.DataFrame) and "macro_f1" in comparison_df.columns
# assert (ARTIFACT_DIR / "model_comparison_results.csv").exists()
# print(f"\u2713 Model comparison: {len(comparison_df)} models evaluated")
# print(f"   Best: {selected_model_name} — macro-F1 {comparison_df.iloc[0]['macro_f1']:.3f}")


# Xiaofei's Task 6 TODO implementation / 中文注释：实现分层交叉验证并完成模型比较。
# 中文注释：导入 clone，确保每个 CV fold 都从未拟合的新模型开始。
from sklearn.base import clone
# 中文注释：导入分层 K 折切分器，维持每折正负类比例。
from sklearn.model_selection import StratifiedKFold
# 中文注释：导入宏平均评估指标，避免多数类准确率掩盖风险类表现。
from sklearn.metrics import f1_score, precision_score, recall_score

# 中文注释：重新定义 evaluate_model，覆盖上方 pass 占位实现。
def evaluate_model(name: str, estimator, X: "pd.DataFrame", y: "pd.Series"):
    # 中文注释：把标签转为连续整数 Series，保证 iloc 和 numpy 索引一致。
    y_series = pd.Series(y).astype(int).reset_index(drop=True)
    # 中文注释：把特征表重置索引，避免原始索引影响 CV 切片。
    X_frame = X.reset_index(drop=True).copy()
    # 中文注释：统计每个类别的样本数，用来决定安全的折数。
    class_count_series = y_series.value_counts().sort_index()
    # 中文注释：读取少数类样本数，避免折数超过任何类别的样本数。
    min_class_count = int(class_count_series.min())
    # 中文注释：如果少数类不足两条样本，则无法执行分层交叉验证。
    if min_class_count < 2:
        # 中文注释：抛出明确错误，提示需要调整标签或样本配置。
        raise ValueError(f"Not enough minority-class samples for stratified CV: {class_count_series.to_dict()}")
    # 中文注释：使用配置折数和少数类样本数中的较小值作为实际 CV 折数。
    n_splits = min(CV_SPLITS, min_class_count)
    # 中文注释：初始化分层 K 折切分器。
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    # 中文注释：创建 out-of-fold 预测数组，默认值用于发现未覆盖样本。
    oof_preds = np.full(len(y_series), -1, dtype=int)
    # 中文注释：初始化每折宏 F1 列表。
    fold_f1s = []
    # 中文注释：初始化每折宏 precision 列表。
    fold_precisions = []
    # 中文注释：初始化每折宏 recall 列表。
    fold_recalls = []
    # 中文注释：遍历每个分层交叉验证 fold。
    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X_frame, y_series), start=1):
        # 中文注释：克隆模型，避免不同 fold 之间共享拟合状态。
        fold_estimator = clone(estimator)
        # 中文注释：在训练 fold 上拟合模型。
        fold_estimator.fit(X_frame.iloc[train_idx], y_series.iloc[train_idx])
        # 中文注释：在测试 fold 上生成预测。
        fold_pred = fold_estimator.predict(X_frame.iloc[test_idx])
        # 中文注释：把当前 fold 的预测写入 out-of-fold 数组。
        oof_preds[test_idx] = fold_pred.astype(int)
        # 中文注释：计算当前 fold 的宏 F1。
        fold_f1 = f1_score(y_series.iloc[test_idx], fold_pred, average="macro", zero_division=0)
        # 中文注释：计算当前 fold 的宏 precision。
        fold_precision = precision_score(y_series.iloc[test_idx], fold_pred, average="macro", zero_division=0)
        # 中文注释：计算当前 fold 的宏 recall。
        fold_recall = recall_score(y_series.iloc[test_idx], fold_pred, average="macro", zero_division=0)
        # 中文注释：保存当前 fold 的宏 F1。
        fold_f1s.append(float(fold_f1))
        # 中文注释：保存当前 fold 的宏 precision。
        fold_precisions.append(float(fold_precision))
        # 中文注释：保存当前 fold 的宏 recall。
        fold_recalls.append(float(fold_recall))
        # 中文注释：打印当前 fold 的模型表现，便于追踪不稳定 fold。
        print(f"{name} fold {fold_id}/{n_splits}: macro-F1={fold_f1:.3f}, precision={fold_precision:.3f}, recall={fold_recall:.3f}")
    # 中文注释：确认每条样本都获得了 out-of-fold 预测。
    if (oof_preds < 0).any():
        # 中文注释：如果有未预测样本，说明 CV 覆盖异常，应立即报错。
        raise RuntimeError(f"OOF prediction coverage failed for {name}")
    # 中文注释：整理当前模型的平均指标和样本规模。
    results = {"name": name, "macro_f1": float(np.mean(fold_f1s)), "macro_precision": float(np.mean(fold_precisions)), "macro_recall": float(np.mean(fold_recalls)), "std_f1": float(np.std(fold_f1s, ddof=0)), "fold_f1s": fold_f1s, "n_samples": int(len(y_series)), "n_positive": int(y_series.sum()), "n_splits": int(n_splits)}
    # 中文注释：返回模型指标和 out-of-fold 预测。
    return results, oof_preds

# 中文注释：从预测表中提取二分类标签。
y_all = feature_frame["risk_label"].astype(int)
# 中文注释：重新初始化所有模型结果列表，覆盖上方占位空列表。
all_results = []
# 中文注释：重新初始化 out-of-fold 预测字典，按模型名称保存预测。
all_oof = {}
# 中文注释：逐个评估前一节定义的模型管线。
for model_name, estimator in model_pipelines:
    # 中文注释：对当前模型执行分层交叉验证评估。
    result_row, oof_pred = evaluate_model(model_name, estimator, feature_frame, y_all)
    # 中文注释：保存当前模型的指标字典。
    all_results.append(result_row)
    # 中文注释：保存当前模型的 out-of-fold 预测。
    all_oof[model_name] = oof_pred
# 中文注释：把模型比较结果整理为 DataFrame，并按宏 F1 从高到低排序。
comparison_df = pd.DataFrame(all_results).sort_values(["macro_f1", "macro_recall", "macro_precision"], ascending=False).reset_index(drop=True)
# 中文注释：保存模型比较 artifact。
save_table(comparison_df, "model_comparison_results.csv")
# 中文注释：优先从非 Dummy 模型中选择最佳模型，避免把多数类基线作为最终解释模型。
non_baseline_comparison = comparison_df[~comparison_df["name"].str.lower().str.contains("dummy")].copy()
# 中文注释：如果存在非基线模型，则选择其中宏 F1 最高者。
if not non_baseline_comparison.empty:
    # 中文注释：读取最佳非基线模型名称。
    selected_model_name = str(non_baseline_comparison.iloc[0]["name"])
# 中文注释：如果只有基线模型，则退回到总体第一名。
else:
    # 中文注释：读取总体第一名模型名称。
    selected_model_name = str(comparison_df.iloc[0]["name"])
# 中文注释：从模型管线列表中取出被选中的 sklearn estimator。
selected_estimator = dict(model_pipelines)[selected_model_name]
# 中文注释：下游诊断继续使用完整 feature_frame，因为管线内部会选择所需列。
selected_X = feature_frame.copy()
# 中文注释：读取被选中模型的 out-of-fold 预测。
selected_pred = all_oof[selected_model_name]
# 中文注释：展示模型比较表，便于报告撰写。
display(comparison_df)
# 中文注释：打印最终选择的模型和宏 F1。
print(f"✓ Model comparison: {len(comparison_df)} models evaluated")
# 中文注释：打印最佳模型摘要。
print(f"   Best non-baseline: {selected_model_name} — macro-F1 {float(non_baseline_comparison.iloc[0]['macro_f1'] if not non_baseline_comparison.empty else comparison_df.iloc[0]['macro_f1']):.3f}")


## 7. Selected Model Diagnostics and Error Analysis

The model comparison picks the highest macro-F1 model. Now we dig deeper:

**Confusion matrix**: Shows true positives, false positives, false negatives, true negatives. With severe class imbalance, expect:
- Many true negatives (correctly predicted non-risk businesses)
- Moderate false positives (predicted risk, actually stable)
- Some false negatives (missed declining restaurants)

**Per-class metrics**:
- **Non-risk F1 ≈ 0.85-0.90**: High precision and recall for the majority class
- **Risk F1 ≈ 0.30-0.40**: Low precision (many false alarms), moderate recall (finds ~half of true declines)

**Error analysis CSV**: Saves all misclassified businesses with:
- `error_type`: false_positive (predicted decline, didn't happen) vs false_negative (missed real decline)
- `historical_text_excerpt`: Students can read what the model "saw" and understand why it failed

**You should ask**: Are false positives or false negatives more costly for a restaurant intelligence system?

In [ ]:
# TODO (Category D — Medium pipeline): Build the confusion matrix and out-of-fold analysis.
# Estimated time: ~1 h
#
# Step 1 — Confusion matrix from oof predictions:
#   cm = confusion_matrix(y_all, selected_pred, labels=[0, 1])
#   cm_df: index = ["actual_non_risk_0", "actual_risk_1"],
#          columns = ["pred_non_risk_0", "pred_risk_1"]
#   Save cm_df to ARTIFACT_DIR / "confusion_matrix_selected_model.csv".
#
# Step 2 — Build error_analysis_cases (formerly oof_df) with per-business detail:
#   Columns to include: business_id, name, city, state, primary_category,
#     historical_review_count, future_review_count, future_review_support_tier,
#     historical_avg_stars, future_avg_stars, quality_decline, risk_label,
#     selected_model (string name), predicted_risk (selected_pred), is_error (bool)
#   Save to ARTIFACT_DIR / "error_analysis_cases.csv".
#
# Step 3 — Build classification_report_df:
#   from sklearn.metrics import classification_report
#   report_dict = classification_report(y_all, selected_pred,
#                   target_names=["non-risk","risk"], output_dict=True)
#   classification_report_df = pd.DataFrame(report_dict).T.reset_index().rename(
#       columns={"index": "class"})
#   Save to ARTIFACT_DIR / "selected_model_classification_report.csv".
#
# Step 4 — Print classification_report(y_all, selected_pred, target_names=["non-risk","risk"]).
#
# Step 5 — Display cm_df.

# TODO: build cm_df from confusion_matrix
cm_df = None  # replace

# TODO: build error_analysis_cases DataFrame
oof_df = None  # replace with error_analysis_cases rows

# TODO: build classification_report_df and save
classification_report_df = None  # replace

# TODO: save cm_df and oof_df; print classification_report; display cm_df

# --- Validation ---
# assert cm_df is not None, "cm_df must be computed"
# assert (ARTIFACT_DIR / "confusion_matrix_selected_model.csv").exists()
# assert oof_df is not None and "is_error" in oof_df.columns
# assert (ARTIFACT_DIR / "error_analysis_cases.csv").exists()
# assert (ARTIFACT_DIR / "selected_model_classification_report.csv").exists()
# print(f"\u2713 Confusion matrix, error analysis, and classification report saved")

# Xiaofei's Task 6 TODO implementation / 中文注释：构建选中模型的混淆矩阵、分类报告和错误分析表。
# 中文注释：导入混淆矩阵和分类报告函数。
from sklearn.metrics import classification_report, confusion_matrix

# 中文注释：基于真实标签和 out-of-fold 预测计算固定标签顺序的混淆矩阵。
cm = confusion_matrix(y_all, selected_pred, labels=[0, 1])
# 中文注释：把混淆矩阵转换为带语义行列名的 DataFrame。
cm_df = pd.DataFrame(cm, index=["actual_non_risk_0", "actual_risk_1"], columns=["pred_non_risk_0", "pred_risk_1"])
# 中文注释：给混淆矩阵索引命名，避免 CSV 中出现 Unnamed: 0。
cm_df.index.name = "actual_label"
# 中文注释：保存选中模型混淆矩阵 artifact。
save_table(cm_df, "confusion_matrix_selected_model.csv", index=True)
# 中文注释：定义错误分析表需要保留的业务和标签字段。
error_columns = ["business_id", "name", "city", "state", "primary_category", "historical_review_count", "future_review_count", "future_review_support_tier", "historical_avg_stars", "future_avg_stars", "quality_decline", "risk_label", "historical_text"]
# 中文注释：只保留当前 feature_frame 中实际存在的字段。
error_columns = [col for col in error_columns if col in feature_frame.columns]
# 中文注释：从预测表复制错误分析基础字段。
oof_df = feature_frame[error_columns].copy()
# 中文注释：记录当前错误分析对应的选中模型名称。
oof_df["selected_model"] = selected_model_name
# 中文注释：写入选中模型的 out-of-fold 风险预测。
oof_df["predicted_risk"] = selected_pred.astype(int)
# 中文注释：标记预测是否与真实风险标签不一致。
oof_df["is_error"] = oof_df["predicted_risk"].astype(int) != oof_df["risk_label"].astype(int)
# 中文注释：用 false_positive / false_negative / correct 描述错误类型。
oof_df["error_type"] = np.select([(oof_df["risk_label"].eq(0) & oof_df["predicted_risk"].eq(1)), (oof_df["risk_label"].eq(1) & oof_df["predicted_risk"].eq(0))], ["false_positive", "false_negative"], default="correct")
# 中文注释：保留历史文本短摘，方便人工检查模型看到的证据。
oof_df["historical_text_excerpt"] = oof_df.get("historical_text", pd.Series("", index=oof_df.index)).fillna("").astype(str).str.slice(0, 500)
# 中文注释：如果错误分析表包含完整历史文本，则移除长文本列以控制 CSV 体积。
if "historical_text" in oof_df.columns:
    # 中文注释：删除完整历史文本，保留上方短摘即可。
    oof_df = oof_df.drop(columns=["historical_text"])
# 中文注释：保存错误分析 artifact。
save_table(oof_df, "error_analysis_cases.csv")
# 中文注释：生成字典形式的分类报告，包含每类 precision、recall 和 F1。
report_dict = classification_report(y_all, selected_pred, target_names=["non-risk", "risk"], output_dict=True, zero_division=0)
# 中文注释：逐行整理分类报告，避免 accuracy 标量在 DataFrame 中被错误扩展。
classification_report_rows = []
# 中文注释：遍历分类报告字典中的每个类别或汇总项。
for report_label, report_values in report_dict.items():
    # 中文注释：如果当前项是字典，则直接展开 precision、recall、f1 和 support。
    if isinstance(report_values, dict):
        # 中文注释：保存当前类别或平均项的指标。
        classification_report_rows.append({"class": report_label, **report_values})
    # 中文注释：如果当前项是 accuracy 标量，则单独整理成一行。
    else:
        # 中文注释：accuracy 没有 per-class precision/recall，保留整体准确率和总样本数。
        classification_report_rows.append({"class": report_label, "precision": np.nan, "recall": np.nan, "f1-score": float(report_values), "support": int(len(y_all))})
# 中文注释：把整理后的分类报告转换为 DataFrame，方便导出和展示。
classification_report_df = pd.DataFrame(classification_report_rows)
# 中文注释：保存选中模型分类报告 artifact。
save_table(classification_report_df, "selected_model_classification_report.csv")
# 中文注释：打印人类可读版本的分类报告。
print(classification_report(y_all, selected_pred, target_names=["non-risk", "risk"], zero_division=0))
# 中文注释：展示混淆矩阵，便于快速查看错误结构。
display(cm_df)
# 中文注释：展示错误案例样例，优先显示误分类行。
display(oof_df.sort_values(["is_error", "error_type"], ascending=[False, True]).head(20))
# 中文注释：打印诊断 artifact 保存完成的信息。
print("✓ Confusion matrix, error analysis, and classification report saved")


## 8. Feature Importance or Coefficients

**For linear models** (Logistic Regression): Extract `model.coef_` to see which text/structured features correlate with risk=1.

**Expected top positive coefficients** (predict quality decline):
- Text terms like "hotel", "buffet", "chipotle", "steak" — these are often **category confounders**, not quality signals
- High `historical_low_star_share` — businesses that already had bad reviews are more likely to decline further

**Expected top negative coefficients** (predict stability):
- Text terms like "beer", "coffee", "sushi", "tacos" — again, cuisine/category markers
- High `historical_avg_stars` — businesses that were already excellent tend to stay excellent

**Critical insight for students**: Top coefficients are **not causal explanations**. "The word 'hotel' causes decline" is wrong — instead, hotels and buffets happen to have different baseline quality distributions in Yelp. This is why Model 0 (category_only_baseline) exists: to quantify category confounding.


In [ ]:
# TODO (Category D — Medium pipeline): Extract and display feature importance or coefficients.
# Estimated time: ~0.5 h
#
# For the selected model:
# 1. Refit the selected estimator on all data (not CV): fitted = clone(selected_estimator).fit(selected_X, y_all)
# 2. Extract feature names using get_feature_names_out() from the ColumnTransformer or TfidfVectorizer.
#    - If the pipeline has a "ct" step: use ct.get_feature_names_out()
#    - If the pipeline has a "tfidf" step: use tfidf.get_feature_names_out()
#    - Otherwise: use generic names.
# 3. Get coefficients or importances:
#    - LogisticRegression: model.coef_[0]
#    - LinearSVC: model.coef_[0]
#    - DummyClassifier / unsupported: skip gracefully.
# 4. Build feature_importance_df: columns [feature, importance] sorted by abs(importance) descending.
# 5. Save to "feature_importance_selected_model.csv".
# 6. Plot top 20 features (barh chart); save to "feature_importance_top20.png".

# TODO: refit selected estimator on full data
fitted_selected = None  # replace

# TODO: extract feature names and coefficients
feature_importance_df = None  # replace

# TODO: save CSV; plot top 20; save PNG

# --- Validation ---
# assert (ARTIFACT_DIR / "feature_importance_selected_model.csv").exists()
# print(f"\u2713 Feature importance saved: {len(feature_importance_df) if feature_importance_df is not None else 0} features")


# Xiaofei's Task 6 TODO implementation / 中文注释：拟合选中模型并导出线性特征重要性。
# 中文注释：导入 clone，确保全量重拟合不污染原 estimator 对象。
from sklearn.base import clone

# 中文注释：克隆选中模型并在全部可用样本上重新拟合。
fitted_selected = clone(selected_estimator).fit(selected_X, y_all)
# 中文注释：读取管线最后的分类器步骤。
final_model = fitted_selected.named_steps.get("clf") if hasattr(fitted_selected, "named_steps") else fitted_selected
# 中文注释：初始化特征名称为 None，后面按管线结构提取。
feature_names = None
# 中文注释：如果模型管线包含 ColumnTransformer，则优先从 ct 步骤读取特征名。
if hasattr(fitted_selected, "named_steps") and "ct" in fitted_selected.named_steps:
    # 中文注释：读取已经拟合的列转换器。
    fitted_ct = fitted_selected.named_steps["ct"]
    # 中文注释：尝试使用 sklearn 的 get_feature_names_out 获取转换后特征名。
    try:
        # 中文注释：获取文本、数值和类别展开后的特征名。
        feature_names = fitted_ct.get_feature_names_out()
    # 中文注释：如果某个子管线不支持特征名输出，则回退到通用名称。
    except Exception:
        # 中文注释：根据转换后矩阵宽度生成 feature_0 这类通用特征名。
        feature_names = np.array([f"feature_{idx}" for idx in range(fitted_ct.transform(selected_X).shape[1])])
# 中文注释：如果模型管线包含直接的 TF-IDF 步骤，则读取词项名称。
elif hasattr(fitted_selected, "named_steps") and "tfidf" in fitted_selected.named_steps:
    # 中文注释：从 TF-IDF 向量器中读取词项特征名。
    feature_names = fitted_selected.named_steps["tfidf"].get_feature_names_out()
# 中文注释：如果最终模型有线性系数，则使用系数表示风险方向和强度。
if hasattr(final_model, "coef_"):
    # 中文注释：读取二分类线性模型的第一行系数。
    importance_values = np.asarray(final_model.coef_[0], dtype=float)
# 中文注释：如果最终模型有树模型重要性，则使用 feature_importances_。
elif hasattr(final_model, "feature_importances_"):
    # 中文注释：读取非线性模型提供的特征重要性。
    importance_values = np.asarray(final_model.feature_importances_, dtype=float)
# 中文注释：如果模型不支持特征重要性，则创建空结果并优雅跳过绘图。
else:
    # 中文注释：使用空数组表示没有可导出的重要性。
    importance_values = np.array([], dtype=float)
# 中文注释：如果前面没有取到特征名，则生成与重要性长度一致的通用名称。
if feature_names is None:
    # 中文注释：生成 feature_0 这类通用特征名。
    feature_names = np.array([f"feature_{idx}" for idx in range(len(importance_values))])
# 中文注释：如果特征名长度和系数长度不一致，则回退为通用名称。
if len(feature_names) != len(importance_values):
    # 中文注释：重新生成与系数长度一致的通用特征名。
    feature_names = np.array([f"feature_{idx}" for idx in range(len(importance_values))])
# 中文注释：将特征名和重要性整理成 DataFrame，并按绝对值降序排列。
feature_importance_df = pd.DataFrame({"feature": feature_names, "importance": importance_values}).assign(abs_importance=lambda df: df["importance"].abs()).sort_values("abs_importance", ascending=False).drop(columns=["abs_importance"]).reset_index(drop=True)
# 中文注释：保存选中模型特征重要性 artifact。
save_table(feature_importance_df, "feature_importance_selected_model.csv")
# 中文注释：如果存在可绘制的重要性，则绘制绝对值最高的 20 个特征。
if not feature_importance_df.empty:
    # 中文注释：取绝对重要性最高的 20 个特征并按原始重要性排序以便水平条形图阅读。
    top_features = feature_importance_df.reindex(feature_importance_df["importance"].abs().sort_values(ascending=False).index).head(20).sort_values("importance")
    # 中文注释：创建特征重要性图。
    plt.figure(figsize=(9, 7))
    # 中文注释：按正负方向设置条形颜色，正值表示更倾向预测 risk=1。
    bar_colors = np.where(top_features["importance"] >= 0, "#b2182b", "#2166ac")
    # 中文注释：绘制水平条形图。
    plt.barh(top_features["feature"], top_features["importance"], color=bar_colors)
    # 中文注释：添加零线，区分正负风险方向。
    plt.axvline(0, color="black", linewidth=0.8)
    # 中文注释：设置图标题。
    plt.title(f"Top coefficients for {selected_model_name}")
    # 中文注释：设置 x 轴标签。
    plt.xlabel("Coefficient / importance toward risk=1")
    # 中文注释：保存特征重要性图。
    save_figure("feature_importance_top20.png")
    # 中文注释：关闭当前图，避免后续图表叠加。
    plt.close()
# 中文注释：如果没有可绘制的重要性，则打印跳过说明。
else:
    # 中文注释：提示当前模型不支持系数或重要性导出。
    print(f"Selected model {selected_model_name} does not expose coefficients or feature importances.")
# 中文注释：展示前 20 个重要特征。
display(feature_importance_df.head(20))
# 中文注释：打印特征重要性保存完成的信息。
print(f"✓ Feature importance saved: {len(feature_importance_df)} features")


## Extension: Sentence-Transformer Embeddings as Features

Bag-of-words and TF-IDF representations capture term frequency but lose word-order
and semantic relatedness. Pretrained sentence embeddings (e.g., `all-MiniLM-L6-v2`)
encode meaning rather than just words.

This section adds a sentence-transformer pipeline to the model comparison. If
`sentence-transformers` is not available, a TF-IDF + SVD fallback is used instead.
Compare its macro-F1 against the classical text models above.

In [ ]:
# TODO (Category A — Modern library): Sentence-transformer embeddings as a classification feature set.
# Estimated time: ~2.5 h
#
# Expected inputs:
#   feature_frame    : DataFrame with at least a text column (e.g., "historical_text")
#   y_all            : Series of binary risk labels
#   evaluate_model   : function from Section 6 (already defined above)
#   CV_SPLITS, RANDOM_SEED : constants from the configuration cell
#
# ── A) Encode text with sentence-transformers (or TF-IDF+SVD fallback) ──────
#
#   Try:  from sentence_transformers import SentenceTransformer
#         model = SentenceTransformer("all-MiniLM-L6-v2")
#         X_emb = model.encode(feature_frame["historical_text"].fillna("").tolist(),
#                              show_progress_bar=False)
#         X_emb_df = pd.DataFrame(X_emb, index=feature_frame.index,
#                                 columns=[f"emb_{i}" for i in range(X_emb.shape[1])])
#
#   Fallback (if sentence_transformers unavailable):
#         from sklearn.feature_extraction.text import TfidfVectorizer
#         from sklearn.decomposition import TruncatedSVD
#         from sklearn.pipeline import Pipeline
#         dim_reducer = Pipeline([
#             ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
#             ("svd",   TruncatedSVD(n_components=64, random_state=RANDOM_SEED)),
#         ])
#         X_emb_arr = dim_reducer.fit_transform(feature_frame["historical_text"].fillna(""))
#         X_emb_df = pd.DataFrame(X_emb_arr, index=feature_frame.index,
#                                 columns=[f"emb_{i}" for i in range(X_emb_arr.shape[1])])
#         representation_label = "tfidf_svd_fallback"
#
# ── B) Build a logistic-regression pipeline on embeddings ───────────────────
#
#   from sklearn.linear_model import LogisticRegression
#   from sklearn.pipeline import Pipeline
#   from sklearn.preprocessing import StandardScaler
#
#   st_pipeline = Pipeline([
#       ("scaler", StandardScaler()),
#       ("clf",    LogisticRegression(class_weight="balanced",
#                                    max_iter=1000, random_state=RANDOM_SEED)),
#   ])
#   model_name = "ST_Logistic" (or "TF-IDF_SVD_Logistic" for the fallback)
#
# ── C) Evaluate with the same evaluate_model function ───────────────────────
#
#   st_results, st_oof = evaluate_model(model_name, st_pipeline, X_emb_df, y_all)
#   Print macro-F1 and compare against the best classical model from comparison_df.
#
# ── D) Save results ──────────────────────────────────────────────────────────
#
#   Save a comparison table (best classical vs. embedding model) to
#   ARTIFACT_DIR / "embedding_vs_classical_comparison.csv".
#
# ── E) Interpret ─────────────────────────────────────────────────────────────
#
#   In 2–4 comments, discuss:
#     Does the embedding model outperform TF-IDF? Why or why not?
#     What might limit embedding features for this small, imbalanced dataset?
#
# Hint: Wrap sentence-transformers import in try/except; fall back transparently.
#       Mark the representation_label clearly in the saved CSV.

try:
    from sentence_transformers import SentenceTransformer as _SentenceTransformer
    _ST_AVAILABLE = True
except ImportError:
    _ST_AVAILABLE = False
    print("sentence-transformers not available; will use TF-IDF + SVD fallback.")

# TODO: encode historical_text into X_emb_df (ST or TF-IDF+SVD fallback)
X_emb_df = None  # replace

# TODO: define st_pipeline (LogisticRegression on embeddings)
st_pipeline = None  # replace

# TODO: call evaluate_model for the embedding pipeline
# st_results, st_oof = evaluate_model("ST_Logistic", st_pipeline, X_emb_df, y_all)

# TODO: compare st_results against comparison_df and save to embedding_vs_classical_comparison.csv
embedding_comparison_df = None  # replace

# --- Validation ---
# assert X_emb_df is not None, "X_emb_df must be computed"
# assert embedding_comparison_df is not None
# assert (ARTIFACT_DIR / "embedding_vs_classical_comparison.csv").exists()
# print(f"\u2713 Embedding model evaluated: {st_results.get('macro_f1', 0):.3f} macro-F1")

# Xiaofei's Task 6 TODO implementation / 中文注释：使用本地 sentence-transformers 或 TF-IDF+SVD fallback 构建 embedding 分类器。
# 中文注释：导入 TruncatedSVD，供本地 fallback 压缩 TF-IDF 表示。
from sklearn.decomposition import TruncatedSVD
# 中文注释：导入逻辑回归，作为 embedding 特征上的分类器。
from sklearn.linear_model import LogisticRegression
# 中文注释：导入 Pipeline，串联 embedding 标准化和分类器。
from sklearn.pipeline import Pipeline
# 中文注释：导入标准化器，让 dense embedding 各维度尺度更适合逻辑回归。
from sklearn.preprocessing import StandardScaler
# 中文注释：导入 TF-IDF 向量器，供 sentence-transformers 不可用时使用。
from sklearn.feature_extraction.text import TfidfVectorizer

# 中文注释：读取历史文本并填补缺失值。
embedding_texts = feature_frame["historical_text"].fillna("").astype(str).tolist()
# 中文注释：读取本地 embedding 模型路径配置。
embedding_model_path = os.environ.get("TASK6_EMBEDDING_MODEL_PATH", "").strip()
# 中文注释：读取是否允许在线下载 embedding 模型的配置。
allow_embedding_download = os.environ.get("TASK6_ALLOW_EMBEDDING_DOWNLOAD", "0") == "1"
# 中文注释：默认使用 TF-IDF+SVD fallback 表示，只有本地模型可用或显式允许下载时才切换。
representation_label = "tfidf_svd_fallback"
# 中文注释：初始化 embedding 数组为空，后面按可用后端填充。
X_emb_arr = None
# 中文注释：判断是否可以尝试 sentence-transformers 编码。
use_sentence_transformer = _ST_AVAILABLE and DATASET_MODE != "smoke_test" and (bool(embedding_model_path and Path(embedding_model_path).exists()) or allow_embedding_download)
# 中文注释：如果满足条件，则优先尝试本地 sentence-transformers 模型。
if use_sentence_transformer:
    # 中文注释：捕获模型加载或编码失败，失败时自动回退到 TF-IDF+SVD。
    try:
        # 中文注释：选择本地模型路径；只有显式允许下载时才使用远程模型 ID。
        model_name_or_path = embedding_model_path if embedding_model_path else "all-MiniLM-L6-v2"
        # 中文注释：加载句向量模型。
        embedding_model = _SentenceTransformer(model_name_or_path)
        # 中文注释：对每个商家的历史评论聚合文本编码。
        X_emb_arr = embedding_model.encode(embedding_texts, batch_size=32, show_progress_bar=False)
        # 中文注释：记录当前使用的向量表示方法。
        representation_label = f"sentence_transformer:{model_name_or_path}"
    # 中文注释：如果 sentence-transformers 失败，则打印原因并进入 fallback。
    except Exception as exc:
        # 中文注释：提示失败原因，便于报告中说明可复现路径。
        print(f"Sentence-transformer embedding failed; using TF-IDF + SVD fallback: {exc}")
        # 中文注释：清空 embedding 数组，触发下方 fallback。
        X_emb_arr = None
        # 中文注释：恢复 fallback 表示标签。
        representation_label = "tfidf_svd_fallback"
# 中文注释：如果没有成功得到句向量，则使用本地 TF-IDF+SVD fallback。
if X_emb_arr is None:
    # 中文注释：拟合 TF-IDF，把历史文本转成稀疏词项矩阵。
    fallback_tfidf = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True)
    # 中文注释：转换全部历史文本。
    fallback_matrix = fallback_tfidf.fit_transform(embedding_texts)
    # 中文注释：根据样本数和词表大小选择安全的 SVD 维度。
    fallback_components = max(2, min(64, fallback_matrix.shape[0] - 1, fallback_matrix.shape[1] - 1))
    # 中文注释：创建 SVD 降维器。
    fallback_svd = TruncatedSVD(n_components=fallback_components, random_state=RANDOM_SEED)
    # 中文注释：得到 dense fallback embedding 数组。
    X_emb_arr = fallback_svd.fit_transform(fallback_matrix)
    # 中文注释：记录 fallback 维度信息。
    representation_label = f"tfidf_svd_fallback_{fallback_components}d"
# 中文注释：确保 embedding 数组是 numpy 数组。
X_emb_arr = np.asarray(X_emb_arr)
# 中文注释：把 embedding 数组转换成 DataFrame，保持与标签相同的索引。
X_emb_df = pd.DataFrame(X_emb_arr, index=feature_frame.index, columns=[f"emb_{idx}" for idx in range(X_emb_arr.shape[1])])
# 中文注释：定义 embedding 上的逻辑回归分类管线。
st_pipeline = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED))])
# 中文注释：根据表示类型命名 embedding 模型。
embedding_model_name = "ST Logistic" if representation_label.startswith("sentence_transformer") else "TF-IDF SVD Logistic"
# 中文注释：使用与经典模型相同的分层 CV 函数评估 embedding 模型。
st_results, st_oof = evaluate_model(embedding_model_name, st_pipeline, X_emb_df, y_all)
# 中文注释：给 embedding 结果补充表示来源说明。
st_results["representation"] = representation_label
# 中文注释：取当前经典模型比较表中的最佳一行作为对照。
best_classical_row = comparison_df.head(1).copy()
# 中文注释：标记经典模型对照组。
best_classical_row["comparison_group"] = "best_classical"
# 中文注释：给经典模型补充表示来源字段，方便和 embedding 结果同表比较。
best_classical_row["representation"] = best_classical_row["name"]
# 中文注释：把 embedding 模型结果转换成单行 DataFrame。
embedding_row = pd.DataFrame([st_results])
# 中文注释：标记 embedding 模型组。
embedding_row["comparison_group"] = "embedding_model"
# 中文注释：合并最佳经典模型和 embedding 模型比较结果。
embedding_comparison_df = pd.concat([best_classical_row, embedding_row], ignore_index=True, sort=False)
# 中文注释：保存 embedding 与经典模型比较 artifact。
save_table(embedding_comparison_df, "embedding_vs_classical_comparison.csv")
# 中文注释：展示 embedding 比较表。
display(embedding_comparison_df[["comparison_group", "name", "representation", "macro_f1", "macro_precision", "macro_recall", "std_f1"]])
# 中文注释：解读：embedding 未必优于 TF-IDF，因为样本量有限且类别不平衡会放大估计方差。
# 中文注释：解读：若 TF-IDF 胜出，通常说明本任务更依赖可解释的词项和类别线索。
# 中文注释：解读：若 embedding 胜出，通常说明语义聚合补充了稀疏词项难以捕捉的相似表达。
# 中文注释：打印 embedding 模型评估完成的信息。
print(f"✓ Embedding model evaluated: {st_results.get('macro_f1', 0):.3f} macro-F1 using {representation_label}")


## Data Mining Concept: Temporal Leakage and Deployment Validity

**Temporal leakage** occurs when information from the future window contaminates the historical features used for prediction. A model trained with leakage may look strong in cross-validation but fail in deployment because the leaked information would not exist at prediction time.

Task 6 prevents leakage by enforcing a cutoff date:

- features use reviews with `review_date < CUTOFF_DATE`;
- labels use reviews with `review_date >= CUTOFF_DATE`;
- aggregate Yelp business metrics that may include future activity are excluded from the model features;
- the leakage audit verifies the historical/future boundary before submission.

The main lesson is that supervised data mining is not just model selection. The feature window, label window, and evaluation protocol define whether the result is meaningful.


## 9. Leakage Audit

This audit verifies that features use only historical data while labels use only future data.

**What we check**:
1. Historical text dates are before the cutoff.
2. Future label dates are on or after the cutoff.
3. Business aggregate stars/review counts from the full Yelp file are not used as model features.
4. No generated explanation artifacts are used as predictive features.

If the audit fails, the notebook raises an error because the model comparison is no longer deployment-valid.


In [ ]:
generated_explanation_cols = [c for c in feature_frame.columns if "explanation" in c.lower()]
max_hist_date = pd.to_datetime(prediction_frame["max_historical_review_date"], errors="coerce").max()
min_future_date = pd.to_datetime(prediction_frame["min_future_review_date"], errors="coerce").min()

leakage_audit = {
    "cutoff_date": str(CUTOFF_DATE.date()),
    "no_future_reviews_used_in_features": bool(pd.isna(max_hist_date) or max_hist_date < CUTOFF_DATE),
    "labels_computed_only_from_future_window": bool(pd.isna(min_future_date) or min_future_date >= CUTOFF_DATE),
    "historical_features_pre_cutoff_only": True,
    "business_aggregate_stars_review_count_excluded": True,
    "generated_explanation_artifacts_used_as_prediction_features": bool(generated_explanation_cols),
    "generated_explanation_feature_columns": generated_explanation_cols,
    "label_config": label_config,
    "main_metric": "macro-F1",
}
if not leakage_audit["no_future_reviews_used_in_features"]:
    raise RuntimeError("Leakage audit failed: a historical feature date is not before the cutoff.")
if not leakage_audit["labels_computed_only_from_future_window"]:
    raise RuntimeError("Leakage audit failed: label dates are not restricted to the future window.")
if generated_explanation_cols:
    raise RuntimeError("Generated explanation artifacts should not be used directly as predictive features in this notebook.")
save_json(leakage_audit, "leakage_audit.json")
print(json.dumps(leakage_audit, indent=2, default=str))


## Visual Evidence and Interpretation

Visuals are evidence, not decoration. Use them to explain class balance, model comparison, and selected-model failure modes.

In [ ]:
# TODO (Category D — Medium pipeline): Plot class balance and per-category error analysis.
# Estimated time: ~0.5 h
#
# Step 1 — Class balance bar chart and CSV:
#   Count risk_label == 0 and risk_label == 1 in feature_frame.
#   Save the counts table to ARTIFACT_DIR / "class_balance.csv"
#   (columns: risk_label, count, label).
#   Plot a bar chart (two bars: non-risk vs. risk).
#   Save to ARTIFACT_DIR / "class_balance.png".
#   Print the counts and the imbalance ratio.
#
# Step 2 — Per-category error rate (requires oof_df):
#   Group oof_df by primary_category: compute error_rate = mean(is_error), businesses = count.
#   Filter to categories with >= 2 businesses.
#   Sort by error_rate descending.
#   Save to "error_rate_by_category.csv".
#   Plot a horizontal bar chart of the top 10 categories by error rate.
#   Save to "error_rate_by_category.png".
#
# Step 3 — In 2–4 comments, write a brief interpretation:
#   Which class is harder to predict? Which categories have the highest error rate?
#   What could explain these patterns?

# TODO: class balance bar chart and CSV
class_counts = feature_frame["risk_label"].value_counts().reset_index().rename(columns={"risk_label":"count","index":"risk_label"}) if "feature_frame" in dir() else pd.DataFrame()
# TODO: save class_balance.csv and plot/save class_balance.png

# TODO: per-category error analysis from oof_df
error_by_category = None  # replace

# TODO: save and plot

# --- Validation ---
# assert (ARTIFACT_DIR / "class_balance.csv").exists()
# assert (ARTIFACT_DIR / "class_balance.png").exists()
# print(f"\u2713 Class balance and error analysis saved")

# Xiaofei's Task 6 TODO implementation / 中文注释：保存类别平衡图和按类别错误率分析。
# 中文注释：按固定标签顺序统计非风险和风险商家数量。
class_counts = feature_frame["risk_label"].value_counts().reindex([0, 1], fill_value=0).rename_axis("risk_label").reset_index(name="count")
# 中文注释：为图表和报告添加可读标签。
class_counts["label"] = class_counts["risk_label"].map({0: "non-risk", 1: "risk"})
# 中文注释：计算总样本数，用于类别比例诊断。
total_businesses = int(class_counts["count"].sum())
# 中文注释：读取非风险样本数。
non_risk_count = int(class_counts.loc[class_counts["risk_label"].eq(0), "count"].iloc[0])
# 中文注释：读取风险样本数。
risk_count = int(class_counts.loc[class_counts["risk_label"].eq(1), "count"].iloc[0])
# 中文注释：计算多数类与少数类的样本数比值。
imbalance_ratio = non_risk_count / max(1, risk_count)
# 中文注释：为每个类别计算样本占比。
class_counts["class_share"] = class_counts["count"] / max(1, total_businesses)
# 中文注释：记录整体风险率，便于报告直接引用。
class_counts["overall_risk_rate"] = risk_count / max(1, total_businesses)
# 中文注释：记录少数类比例，便于检查是否达到建模可行性阈值。
class_counts["minority_class_rate"] = min(non_risk_count, risk_count) / max(1, total_businesses)
# 中文注释：记录非风险与风险样本数量比，便于说明类别不平衡程度。
class_counts["nonrisk_to_risk_ratio"] = imbalance_ratio
# 中文注释：保存类别平衡表，覆盖前面没有完整比例诊断的初始版本。
save_table(class_counts, "class_balance.csv")
# 中文注释：创建类别平衡柱状图。
plt.figure(figsize=(6, 4))
# 中文注释：绘制非风险与风险样本数量。
plt.bar(class_counts["label"], class_counts["count"], color=["#4c78a8", "#e45756"])
# 中文注释：设置图标题。
plt.title("Task 6 class balance")
# 中文注释：设置 y 轴标签。
plt.ylabel("Businesses")
# 中文注释：保存类别平衡图。
save_figure("class_balance.png")
# 中文注释：关闭当前图，避免后续图表叠加。
plt.close()
# 中文注释：打印类别数量和不平衡比例。
print(f"Class counts: non-risk={non_risk_count}, risk={risk_count}, imbalance ratio={imbalance_ratio:.2f}:1")
# 中文注释：如果错误分析表可用，则按一级类别汇总错误率。
if "oof_df" in globals() and isinstance(oof_df, pd.DataFrame) and not oof_df.empty:
    # 中文注释：按 primary_category 计算业务数、错误数、错误率和风险率。
    error_by_category = oof_df.groupby("primary_category", dropna=False).agg(businesses=("business_id", "count"), errors=("is_error", "sum"), error_rate=("is_error", "mean"), risk_rate=("risk_label", "mean")).reset_index()
    # 中文注释：只保留至少两个商家的类别，避免单样本类别误导解释。
    error_by_category = error_by_category[error_by_category["businesses"] >= 2].sort_values(["error_rate", "businesses"], ascending=[False, False]).reset_index(drop=True)
# 中文注释：如果错误分析表不可用，则创建稳定 schema 的空结果。
else:
    # 中文注释：创建空的按类别错误率表。
    error_by_category = pd.DataFrame(columns=["primary_category", "businesses", "errors", "error_rate", "risk_rate"])
# 中文注释：保存按类别错误率 artifact。
save_table(error_by_category, "error_rate_by_category.csv")
# 中文注释：如果存在可绘制的类别错误率，则绘制前十个高错误率类别。
if not error_by_category.empty:
    # 中文注释：选择错误率最高的十个类别并倒序显示。
    top_error_categories = error_by_category.head(10).sort_values("error_rate")
    # 中文注释：创建类别错误率图。
    plt.figure(figsize=(8, 5))
    # 中文注释：绘制水平条形图。
    plt.barh(top_error_categories["primary_category"].astype(str), top_error_categories["error_rate"], color="#7f7f7f")
    # 中文注释：设置 x 轴标签。
    plt.xlabel("Out-of-fold error rate")
    # 中文注释：设置图标题。
    plt.title("Highest error-rate categories")
    # 中文注释：保存类别错误率图。
    save_figure("error_rate_by_category.png")
    # 中文注释：关闭当前图，避免图表叠加。
    plt.close()
# 中文注释：如果没有足够类别可绘图，则打印跳过提示。
else:
    # 中文注释：提示类别错误率图没有足够数据。
    print("Skipping error_rate_by_category.png: no category has at least 2 businesses.")
# 中文注释：解读：risk 类通常更难预测，因为它样本少且未来下滑标签噪声更高。
# 中文注释：解读：高错误率类别往往是样本数较少或历史评论文本更异质的类别。
# 中文注释：解读：类别错误率应用于发现模型盲区，而不应被解释为某类餐厅必然更差。
# 中文注释：展示类别平衡表。
display(class_counts)
# 中文注释：展示错误率最高的类别。
display(error_by_category.head(10))
# 中文注释：打印可视化 artifact 保存完成的信息。
print("✓ Class balance and error analysis saved")


**Insights! 💡**

Task 6 connects classical tabular prediction and text mining.

**Statistical learning era:** logistic regression and Naive Bayes made text and structured-feature classification practical.

**Text mining era:** TF-IDF turned documents into sparse numeric features that work surprisingly well with linear classifiers.

**Modern ML discipline:** the most important improvement is not always a more complex model. In small, imbalanced, temporally structured datasets, careful label definition, leakage control, macro-F1 evaluation, and error analysis often matter more than algorithm complexity.


## Artifact Validation

In [ ]:
required_artifacts = [
    "business_prediction_frame.csv",
    "class_balance.csv",
    "model_comparison_results.csv",
    "selected_model_classification_report.csv",
    "confusion_matrix_selected_model.csv",
    "error_analysis_cases.csv",
    "leakage_audit.json",
]
missing = [f for f in required_artifacts if not (ARTIFACT_DIR / f).exists()]
if missing:
    print("⚠  Missing required artifacts:", missing)
else:
    print("✓ All required artifacts present.")
for f in required_artifacts:
    p = ARTIFACT_DIR / f
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status}: {f}")

## Required Report Deliverables

Your Task 6 report should be approximately **2–3 pages** and include:

1. The cutoff date choice and how the historical/future windows are defined.
2. Class imbalance analysis and why macro-F1 is the right metric here.
3. Comparison of metadata-only, text-only, and combined models.
4. Confusion matrix analysis: what type of errors does the best model make?
5. Leakage audit: what temporal leakage risks remain?
6. One interpretation of the most important features or surprising failure cases.
7. **LSA/SVD comparison**: did the compressed LSA representation improve macro-F1 compared with sparse TF-IDF (M3 vs. M4)? Why might dimensionality reduction help or hurt in this leakage-safe prediction context?

## Common pitfalls to avoid

- Treating accuracy as the main metric when classes are imbalanced.
- Using future-period information in historical features.
- Comparing models on different folds or different labeled subsets without explanation.
- Reporting a complex model as better without macro-F1, per-class metrics, and error analysis.
- Making causal or policy claims from predictive review-text signals.
